# Assignment: Capacitated Facility Location Problem (CFLP)

## Comparison of Exact vs Heuristic Approaches

**Course:** CS616 - Optimization Algorithms  
**Instructors:** Dr. Mahdi Khemakhem & Dr. Essra Aldessouki

---

## 1. Problem Description:

A company must decide which facilities to open to serve a set of clients at minimum total cost. Each facility has a **fixed opening cost** and a **limited capacity**. Each client has a **demand** that must be satisfied. Each client must be **assigned to exactly one open facility**, and the **total demand assigned to any facility cannot exceed its capacity**.

The total cost includes both **facility opening costs** and **client-to-facility assignment costs**.

The **Capacitated Facility Location Problem (CFLP)** is more challenging than the uncapacitated version because it adds capacity constraints, making it significantly harder for exact solvers while creating more realistic scenarios.

The CFLP appears in many real-world applications, such as:
- Warehouse location with storage limits
- Data center placement with processing capacity
- Service center allocation with staff limits
- Distribution hub planning with throughput constraints
- Healthcare facility location with patient capacity
- Retail store placement with inventory limits
- Supply chain network design with capacity restrictions

The global decision of a CFLP is to determine which facilities to open and how to assign clients to these facilities to minimize the total cost while respecting capacity constraints.

The CFLP is **strongly NP-hard**, making it computationally more challenging than the uncapacitated version. Exact methods like Mixed-Integer Programming (MIP) struggle with even moderate-sized instances, while metaheuristics (e.g., Variable Neighborhood Search (VNS)) can provide good solutions in reasonable time.

The CFLP can be formulated as follows:

### Mathematical Formulation

To model the Capacitated Facility Location Problem, we define the following components:

#### Sets:
- $𝐼 = \{1, 2, \ldots, n\}$: Set of $n$ clients.
- $𝐽 = \{1, 2, \ldots, m\}$: Set of potential $m$ facilities.

#### Parameters:
- $f_j$: Fixed cost of opening facility $j \in J$.
- $c_{ij}$: Cost of assigning client $i \in I$ to facility $j \in J$.
- $q_j$: **Capacity** of facility $j \in J$ (maximum demand it can serve).
- $d_i$: **Demand** of client $i \in I$ (must be satisfied).

#### Decision Variables:
$$\forall i \in I, \forall j \in J, x_{𝑖𝑗} =
\begin{cases}
1 & \text{ if client } 𝑖 \text{ is assigned to facility } 𝑗 \\
0 & \text{ otherwise}
\end{cases}
$$
$$\forall j \in J, 𝑦_{𝑗} = \begin{cases}
1 & \text{ if facility } 𝑗 \text{ is opened} \\
0 & \text{ otherwise}
\end{cases}$$

#### Objective Function:

Minimize the total cost:

$$\text{Minimize } Z = \sum_{j \in J} f_j y_j + \sum_{i \in I} \sum_{j \in J} c_{ij} x_{ij}$$

#### Constraints:
1. **Demand satisfaction**: Each client is assigned to exactly one facility:
    $$\sum_{j \in J} x_{ij} = 1, \quad \forall i \in I$$

2. **Linking constraint**: Clients can only be assigned to open facilities:
   $$x_{ij} \leq y_{j}, \quad \forall i \in I, \forall j \in J$$

3. **Capacity constraint**: Total demand assigned to a facility cannot exceed its capacity:
   $$\sum_{i \in I} d_i \cdot x_{ij} \leq q_j \cdot y_j, \quad \forall j \in J$$

4. **Binary constraints** on decision variables:
   $$x_{ij} \in \{0, 1\}, \quad \forall i \in I, \forall j \in J$$
   $$y_{j} \in \{0, 1\}, \quad \forall j \in J$$

#### Problem Complexity:
The capacity constraints make CFLP significantly harder than uncapacitated FLP because:
- **Tighter LP relaxation**: Capacity constraints create stronger linear programming bounds
- **Combinatorial explosion**: Limited capacity forces more facilities to open
- **Packing problem**: Assigning clients becomes a bin-packing subproblem
- **Infeasibility risk**: Tight capacities can lead to infeasible partial solutions

This makes CFLP an excellent benchmark for comparing exact and heuristic methods!

---

## 2. Import Libraries

In this section we import the necessary libraries for data handling, optimization, and visualization.

If a library is not installed, the script will attempt to install it using `pip install`.

In [ ]:
# Try to import necessary libraries, install if not available
try:
    import numpy as np
except ImportError:
    import sys
    !{sys.executable} -m pip install numpy
    import numpy as np
NUMPY_AVAILABLE = True

try:
    import pandas as pd
except ImportError:
    import sys
    !{sys.executable} -m pip install pandas
    import pandas as pd
PANDAS_AVAILABLE = True

try:
    import matplotlib.pyplot as plt
except ImportError:
    import sys
    !{sys.executable} -m pip install matplotlib
    import matplotlib.pyplot as plt
MATPLOTLIB_AVAILABLE = True

try:
    import seaborn as sns
except ImportError:
    import sys
    !{sys.executable} -m pip install seaborn
    import seaborn as sns
SEABORN_AVAILABLE = True

try:
    from scipy.interpolate import make_interp_spline
except ImportError:
    import sys
    !{sys.executable} -m pip install scipy
    from scipy.interpolate import make_interp_spline
SCIPY_AVAILABLE = True

try:
    import gurobipy as gp
    from gurobipy import GRB
except ImportError:
    import sys
    !{sys.executable} -m pip install gurobipy
    import gurobipy as gp
    from gurobipy import GRB
GUROBI_AVAILABLE = True

import time 
import os
import warnings
import shutil 
import json
from typing import Tuple, List

warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

if NUMPY_AVAILABLE and PANDAS_AVAILABLE and MATPLOTLIB_AVAILABLE and SEABORN_AVAILABLE and SCIPY_AVAILABLE and GUROBI_AVAILABLE:
    print("✅ All libraries imported successfully.")
else:
    print("❌ Some libraries failed to import.")


---

## 3. Instance Generator and Loader

In this section, we generate challenging **Capacitated Facility Location Problem (CFLP)** instances that create meaningful computational challenges for both exact and heuristic algorithms. Our instance generator creates **144 test cases** across 3 size categories and 3 difficulty levels, following a validated economic model with realistic capacity constraints.

### 3.1 Instance Generation Methodology

The generator implements a **research-based ratio model** for CFLP following operations research best practices:

$$\sum_{j} f_j = \alpha \times \sum_{i} \min_{j} c_{ij}$$

where:
- $\sum_{j} f_j$ = total fixed costs across all facilities
- $\alpha$ = fixed-to-**optimal**-service cost **ratio** (NOT multiplier!)
- $\sum_{i} \min_{j} c_{ij}$ = **optimal service cost** (minimum cost per client, NOT sum of all pairs!)
- Individual facility costs: $f_j = \frac{\alpha \cdot \text{optimal\_service}}{n} \cdot \frac{\bar{c}_j}{\bar{c}_{global}} \cdot \eta_j$

**CRITICAL**: α is computed against the **optimal service cost** (what you'd pay if each client uses their nearest facility), NOT the sum of all service cost pairs. This ensures proper difficulty calibration!

**Key Insight**: Computational hardness comes from **balanced trade-offs**, not high absolute costs!
- When $\alpha \approx 0.5$: Fixed costs = 50% of optimal service → Open many facilities → **EASY**
- When $\alpha \approx 1.0$: Fixed costs = 100% of optimal service → True trade-offs → **HARDEST!**
- When $\alpha \approx 3.0$: Fixed costs = 300% of optimal service → Open few facilities → **MEDIUM**

This creates a **U-shaped difficulty curve** where moderate instances (α ≈ 1.0) are the most challenging.

### 3.2 Size Categories (Optimized for Free Gurobi + CFLP Hardness)

Instances are carefully sized to **stay within free Gurobi limits (2000 variables, 2000 constraints)** while maximizing computational difficulty through **capacity constraints**:

**Variable Count**: $n \times m + m$ (assignment variables + facility variables)
**Constraint Count**: $n + n \times m + m$ (demand + linking + capacity constraints)

| Category | Facilities | Clients | Variables | Constraints | Characteristics |
|----------|------------|---------|-----------|-------------|-----------------|
| **Small** | 15-20 | 50-80 | 750-1,620 | 815-1,700 | Solvable but challenging with capacity |
| **Medium** | 18-24 | 60-75 | 1,080-1,824 | 1,158-1,923 | Very challenging with tight capacities |
| **Large** | 20-28 | 65-70 | 1,320-1,988 | 1,405-2,086 | **Near limit**, capacity creates extreme difficulty |

**Key Design Principles**:
- **Capacity tightness** creates computational hardness (not just size!)
- Free Gurobi can handle all instances (no license restrictions)
- CFLP with tight capacities is **significantly harder** than larger uncapacitated FLP
- Capacity constraints force **more facilities to open** → harder combinatorial decisions

*Note: Smaller sizes with tight capacities are MORE challenging than larger uncapacitated problems!*

### 3.3 Difficulty Levels (Capacity Tightness + Cost Ratio Model)

Difficulty is controlled through **two mechanisms**:
1. **α ratio** (fixed-to-service cost ratio)
2. **Capacity tightness ratio** (β) - **NEW for CFLP!**

| Level | α Range | β (Capacity/Demand) | Algorithmic Challenge |
|-------|---------|---------------------|----------------------|
| **Easy** | [0.3, 0.6] | [1.4, 1.7] | Loose capacity + cheap fixed → Simple |
| **Moderate** | [0.8, 1.5] | [1.1, 1.3] | **HARDEST! Tight capacity + balanced costs** |
| **Expensive** | [2.0, 4.0] | [1.2, 1.5] | Expensive facilities + moderate capacity |

**Capacity Tightness Ratio (β)**:
$$\beta = \frac{\sum_{j} q_j}{\sum_{i} d_i}$$

- **β < 1.2**: Very tight → forces many facilities open → **EXTREMELY HARD**
- **β ≈ 1.3**: Moderate → interesting trade-offs → **HARD**
- **β > 1.5**: Loose → capacity less binding → easier

**Scientific Basis**: CFLP research shows capacity tightness is the PRIMARY hardness factor:
- Tight capacities create **knapsack-like subproblems** at each facility
- Forces **more facilities to open** → larger integer search space
- Creates **stronger LP bounds** → exact methods struggle more
- **Capacity + cost balance** at β ≈ 1.2, α ≈ 1.0 is HARDEST

**Expected Performance** (CFLP with tight capacities):
- **Gurobi**: 5-30s (small), 30-120s (medium), 60-300s (large) - capacity makes it MUCH harder!
- **VNS**: 2-5% gap (easy), 8-18% gap (moderate), 4-10% gap (expensive)
- **Key insight**: Even small CFLP instances are harder than large uncapacitated FLP!
- **Capacity constraints** create the difficulty, not just problem size!

### 3.4 Instance Distribution

The generator creates **144 total instances** (corrected from 150):
- 48 instances per size category (small, medium, large)
- Within each size: 16 instances per difficulty level
- Ensures balanced evaluation across all 9 combinations (3×3)

### 3.5 Output Format

Each instance is saved with comprehensive metadata:

1. **JSON file** (`{ID}_flp_{size}_{difficulty}_fac{m}_cli{n}.json`):
   - Cost matrices (`fixed_costs`, `service_costs`)
   - Coordinates (`client_coordinates`, `facility_coordinates`)
   - Metadata (`size`, `difficulty`, `alpha`)

2. **PNG visualization** (150 DPI):
   - Client locations (blue circles)
   - Facility locations (red squares)
   - Instance parameters in title

3. **Summary Excel** (`instances_summary.xlsx`) with 4 sheets:
   - All Instances: Complete dataset
   - Size Statistics: Aggregated by size
   - Difficulty Statistics: Aggregated by difficulty
   - Distribution Matrix: Instance counts per category

### 3.6 Key Functions

- **`generate_fixed_costs_from_cost_matrix()`**: Implements the cost model formula using **optimal service cost** as baseline (critical for proper difficulty calibration!)
- **`generate_instances()`**: Creates the full instance suite with scientifically calibrated difficulty
- **`load_flp_instance()`**: Loads instance data from JSON for experimentation

### 3.7 Expected Computational Behavior

With **CFLP + tight capacity constraints** (β ≈ 1.2 for moderate):

**Gurobi (Exact Method)**:
- Small easy: 2-10 seconds (loose capacity, cheap facilities)
- Small moderate: **10-60 seconds** (tight capacity creates extreme difficulty!)
- Medium moderate: **40-180 seconds** (capacity + cost balance = VERY HARD!)
- Large moderate: **120-300+ seconds** (near free Gurobi limits, capacity binding)

**VNS (Heuristic Method)**:
- Small: 0.5-2 seconds (fast but capacity makes quality harder)
- Medium: 1-4 seconds (capacity constraints challenge local search)
- Large: 2-8 seconds (feasibility checks dominate time)
- Gap pattern influenced by **capacity tightness**:
  - Easy (β=1.5, α=0.5): 2-5% gap (loose capacity → easier)
  - **Moderate (β=1.2, α=1.0): 8-18% gap (HARDEST! Tight capacity + cost balance)**
  - Expensive (β=1.3, α=3.0): 4-10% gap (expensive facilities + moderate capacity)

**Why CFLP is Harder**:
1. **Capacity constraints** create knapsack subproblems → NP-hard within NP-hard!
2. **Tighter LP bounds** → branch-and-bound explores more nodes
3. **Feasibility checking** → every move must verify capacity
4. **More facilities needed** → larger combinatorial space to explore

**Key Insight**: **Capacity tightness** (β ≈ 1.2) combined with **cost balance** (α ≈ 1.0) creates the HARDEST instances. Even small CFLP (20 facilities, 70 clients) with tight capacities is harder than large uncapacitated FLP (100 facilities, 500 clients).

**Free Gurobi Compliance**: All instances stay within 2000 variables and 2000 constraints, making the assignment accessible to all students while maintaining significant computational challenge through capacity constraints.

The resulting CFLP instances are suitable for **meaningful algorithm comparison** and provide **scientifically sound computational challenges** that test both solution quality and constraint handling capabilities.

In [ ]:
def generate_fixed_costs_from_cost_matrix(service_costs: np.ndarray, alpha: float, 
                                         noise_range: tuple = (0.8, 1.2)) -> np.ndarray:
    """
    Generate fixed costs using α as RATIO of OPTIMAL service cost to fixed cost.
    
    CRITICAL: α is computed against OPTIMAL service cost (minimum per client),
    NOT against the sum of all service cost pairs!
    
    Research-based model: Total fixed costs should be α × optimal service costs
    This creates meaningful trade-offs:
        - α < 0.5: Fixed costs cheap → open many facilities → EASY
        - α ≈ 1.0: Fixed costs ≈ optimal service costs → hard trade-offs → HARDEST!
        - α > 2.0: Fixed costs expensive → open few facilities → MEDIUM
    
    Formula: 
        optimal_service = Σ(min_j c_ij)  ← Cost if each client uses nearest facility
        target_fixed = α × optimal_service
        f_j = (target_fixed / n) × (c̄_j / c̄_global) × η_j
    
    where:
        - α is the fixed-to-optimal-service cost ratio
        - optimal_service is the minimum achievable service cost
        - n is number of facilities (distributes total budget)
        - (c̄_j / c̄_global) scales by relative facility importance
        - η_j adds heterogeneity noise
    
    Parameters:
    -----------
    service_costs : np.ndarray
        Service cost matrix of shape (num_clients, num_facilities)
    alpha : float
        Fixed-to-optimal-service cost RATIO (0.3-0.6: easy, 0.8-1.5: hard, 2.0-4.0: medium)
    noise_range : tuple
        Range for multiplicative noise (default: ±20%)
    
    Returns:
    --------
    fixed_costs : np.ndarray
        Fixed costs for each facility
    """
    num_clients, num_facilities = service_costs.shape
    
    # CRITICAL FIX: Use OPTIMAL service cost (minimum per client) as baseline!
    # Not the sum of all pairs which is irrelevant to the actual problem
    optimal_service_cost = service_costs.min(axis=1).sum()
    
    # Target: Total fixed costs = α × OPTIMAL service cost
    # This is the CORRECT way to calibrate difficulty!
    target_total_fixed_cost = alpha * optimal_service_cost
    
    # Compute average cost per facility (for proportional distribution)
    avg_cost_per_facility = service_costs.mean(axis=0)
    global_avg_cost = avg_cost_per_facility.mean()
    
    # Distribute total budget proportionally to facility importance
    # Facilities serving expensive regions get higher fixed costs
    importance_weights = avg_cost_per_facility / global_avg_cost
    base_fixed_cost = target_total_fixed_cost / num_facilities
    
    # Scale by importance and add heterogeneity
    noise_factors = np.random.uniform(noise_range[0], noise_range[1], size=num_facilities)
    fixed_costs = base_fixed_cost * importance_weights * noise_factors
    
    # Normalize to maintain target total (compensate for noise)
    fixed_costs = fixed_costs * (target_total_fixed_cost / fixed_costs.sum())
    
    # Ensure positive and reasonable values
    fixed_costs = np.maximum(fixed_costs, 1.0)
    
    return fixed_costs


def generate_instances(folder: str = "cflp_instances", num_total_instances: int = 144, seed: int = 42):
    """
    Generate Capacitated Facility Location Problem (CFLP) instances with scientific rigor.
    
    Creates instances across 3 size categories (small, medium, large) and 
    3 difficulty levels (easy, moderate, expensive), following the methodology:
    
    Size Categories:
    - Small: 2-5 facilities, 30-60 clients
    - Medium: 6-10 facilities, 80-150 clients  
    - Large: 12-20 facilities, 200-400 clients
    
    Difficulty Levels (via α parameter):
    - Easy: α ∈ [5, 10] - Low fixed costs, most facilities profitable
    - Moderate: α ∈ [10, 20] - Balanced trade-offs
    - Expensive: α ∈ [20, 40] - High fixed costs, selective openings
    
    Parameters:
    -----------
    folder : str
        Directory to save generated instances
    num_total_instances : int
        Total number of instances to generate (distributed evenly)
    seed : int
        Random seed for reproducibility
    
    Returns:
    --------
    None (instances saved to disk with summary Excel file)
    """
    np.random.seed(seed)
    
    # ==================================================================================
    # CLEANUP: Remove all old instances before generating new ones
    # ==================================================================================
    print(f"\n{'='*80}")
    print(f"CLEANING UP OLD INSTANCES")
    print(f"{'='*80}")
    
    if os.path.exists(folder):
        file_count = 0
        dir_count = 0
        
        # Count and delete all contents
        for filename in os.listdir(folder):
            file_path = os.path.join(folder, filename)
            try:
                if os.path.isfile(file_path):
                    os.unlink(file_path)
                    file_count += 1
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
                    dir_count += 1
            except Exception as e:
                print(f"⚠️  Warning: Could not delete {file_path}: {e}")
        
        # Remove and recreate folder to ensure complete cleanup
        try:
            shutil.rmtree(folder)
            print(f"✓ Removed existing folder '{folder}'")
            print(f"  - Deleted {file_count} file(s)")
            print(f"  - Deleted {dir_count} subdirectory(ies)")
        except Exception as e:
            print(f"⚠️  Warning: Could not remove folder {folder}: {e}")
    else:
        print(f"✓ Folder '{folder}' does not exist (first run)")
    
    # Create fresh output folder
    os.makedirs(folder, exist_ok=True)
    print(f"✓ Created fresh folder '{folder}'\n")

    # ==================================================================================
    # CFLP DIFFICULTY MODEL (Capacity Tightness + Cost Ratio)
    # ==================================================================================
    # Key factors creating computational hardness in CFLP:
    # 1. CAPACITY TIGHTNESS (β): Ratio of total capacity to total demand
    #    - β < 1.2: Very tight → forces many facilities → EXTREMELY HARD
    #    - β ≈ 1.3: Moderate → interesting trade-offs → HARD
    #    - β > 1.5: Loose → capacity less binding → easier
    # 2. COST RATIO (α): Fixed costs relative to service costs
    # 3. PROBLEM SIZE: Limited by free Gurobi (2000 vars, 2000 constraints)
    #
    # Variables: n*m + m  |  Constraints: n + n*m + m
    # Must keep: (n*m + m) ≤ 2000  AND  (n + n*m + m) ≤ 2000
    size_settings = {
        "small": {
            "fac_range": (15, 21),      # 15-20 facilities
            "cli_range": (50, 81),      # 50-80 clients → Max ~1,620 vars, ~1,700 constraints
            "alpha_ranges": {
                "easy": (0.3, 0.6),      # Cheap facilities + loose capacity
                "moderate": (0.8, 1.5),  # Balanced costs + tight capacity → HARDEST!
                "expensive": (2.0, 4.0)  # Expensive facilities + moderate capacity
            },
            "beta_ranges": {
                "easy": (1.4, 1.7),      # Loose capacity (40-70% slack)
                "moderate": (1.1, 1.3),  # Tight capacity (10-30% slack) → HARDEST!
                "expensive": (1.2, 1.5)  # Moderate capacity
            }
        },
        "medium": {
            "fac_range": (18, 25),      # 18-24 facilities
            "cli_range": (60, 76),      # 60-75 clients → Max ~1,824 vars, ~1,923 constraints
            "alpha_ranges": {
                "easy": (0.3, 0.6),
                "moderate": (0.8, 1.5),  # HARDEST with tight capacity!
                "expensive": (2.0, 4.0)
            },
            "beta_ranges": {
                "easy": (1.4, 1.7),
                "moderate": (1.1, 1.3),  # Very challenging!
                "expensive": (1.2, 1.5)
            }
        },
        "large": {
            "fac_range": (20, 28),      # 20-27 facilities (reduced max to stay within limits)
            "cli_range": (65, 69),      # 65-68 clients → Max ~1,863 vars, ~1,944 constraints (safe margin!)
            "alpha_ranges": {
                "easy": (0.3, 0.6),
                "moderate": (0.8, 1.5),  # HARDEST! Near Gurobi free limit + tight capacity
                "expensive": (2.0, 4.0)
            },
            "beta_ranges": {
                "easy": (1.4, 1.7),
                "moderate": (1.1, 1.3),  # Extremely challenging!
                "expensive": (1.2, 1.5)
            }
        }
    }

    # Calculate instances per category (evenly distributed)
    num_sizes = len(size_settings)
    num_difficulties = 3
    instances_per_combination = num_total_instances // (num_sizes * num_difficulties)
    
    instances_info = []
    instance_id = 1

    print(f"Generating {num_total_instances} CFLP instances...")
    print(f"Distribution: {instances_per_combination} per size-difficulty combination")
    print(f"All instances comply with free Gurobi limits (≤2000 vars, ≤2000 constraints)\n")
    
    # Generate instances for each size-difficulty combination
    for size, params in size_settings.items():
        for difficulty, alpha_range in params["alpha_ranges"].items():
            beta_range = params["beta_ranges"][difficulty]
            
            for rep in range(instances_per_combination):
                # Randomize problem dimensions
                num_facilities = np.random.randint(*params["fac_range"])
                num_clients = np.random.randint(*params["cli_range"])
                
                # Verify free Gurobi compliance
                num_vars = num_clients * num_facilities + num_facilities
                num_constraints = num_clients + num_clients * num_facilities + num_facilities
                
                if num_vars > 2000 or num_constraints > 2000:
                    print(f"⚠️  WARNING: Instance too large! Vars={num_vars}, Constraints={num_constraints}")
                    continue
                
                # Sample α (cost ratio) and β (capacity tightness) from difficulty range
                alpha = np.random.uniform(*alpha_range)
                beta = np.random.uniform(*beta_range)

                # Generate UNIQUE random 2D coordinates using grid-based approach
                # This ensures no overlapping points and creates diverse spatial patterns
                grid_size = 1000  # Large grid for unique positions
                
                # Generate unique client positions
                clients = []
                used_positions = set()
                attempts = 0
                while len(clients) < num_clients and attempts < num_clients * 10:
                    x = np.random.uniform(0, grid_size)
                    y = np.random.uniform(0, grid_size)
                    # Round to ensure uniqueness but maintain diversity
                    pos_key = (round(x, 1), round(y, 1))
                    if pos_key not in used_positions:
                        clients.append([x, y])
                        used_positions.add(pos_key)
                    attempts += 1
                clients = np.array(clients)
                
                # Generate unique facility positions (separate from clients)
                facilities = []
                attempts = 0
                while len(facilities) < num_facilities and attempts < num_facilities * 10:
                    x = np.random.uniform(0, grid_size)
                    y = np.random.uniform(0, grid_size)
                    pos_key = (round(x, 1), round(y, 1))
                    if pos_key not in used_positions:
                        facilities.append([x, y])
                        used_positions.add(pos_key)
                    attempts += 1
                facilities = np.array(facilities)

                # Compute service costs: distance + perturbation
                # Perturbation creates problem structure, not difficulty directly
                service_costs = np.zeros((num_clients, num_facilities))
                
                # Use CONSTANT moderate noise across all difficulties
                # Difficulty comes from α (cost ratio) and problem size, NOT noise!
                base_noise = 0.15  # 15% noise is realistic for real-world problems
                
                for i in range(num_clients):
                    for j in range(num_facilities):
                        euclidean_dist = np.linalg.norm(clients[i] - facilities[j])
                        # Add proportional noise (15% of distance)
                        perturbation = euclidean_dist * np.random.uniform(0, base_noise)
                        service_costs[i, j] = euclidean_dist + perturbation

                # ===================================================================
                # GENERATE DEMANDS AND CAPACITIES FOR CFLP
                # ===================================================================
                # Client demands: uniform random in [5, 20] (realistic range)
                client_demands = np.random.uniform(5, 20, size=num_clients)
                total_demand = client_demands.sum()
                
                # Facility capacities based on β (capacity tightness ratio)
                # β = total_capacity / total_demand
                # Lower β → tighter capacity → harder problem!
                target_total_capacity = beta * total_demand
                
                # Distribute capacity across facilities with heterogeneity
                # Base capacity per facility
                base_capacity = target_total_capacity / num_facilities
                
                # Add heterogeneity: some facilities larger, some smaller
                capacity_factors = np.random.uniform(0.7, 1.3, size=num_facilities)
                facility_capacities = base_capacity * capacity_factors
                
                # Normalize to maintain exact β ratio
                facility_capacities = facility_capacities * (target_total_capacity / facility_capacities.sum())
                
                # Ensure all capacities can serve at least 2-3 clients
                min_capacity = 2 * client_demands.mean()
                facility_capacities = np.maximum(facility_capacities, min_capacity)
                
                # Recalculate actual β after adjustments
                actual_beta = facility_capacities.sum() / total_demand
                
                # ===================================================================
                # GENERATE FIXED COSTS USING α RATIO MODEL
                # ===================================================================
                fixed_costs = generate_fixed_costs_from_cost_matrix(
                    service_costs, 
                    alpha=alpha, 
                    noise_range=(0.8, 1.2)  # ±20% heterogeneity in fixed costs
                )

                # Create instance filename
                instance_name = f"{instance_id:03d}_cflp_{size}_{difficulty}_fac{num_facilities}_cli{num_clients}.json"
                instance_path = os.path.join(folder, instance_name)

                # Prepare instance data with capacities and demands
                instance_data = {
                    "num_facilities": int(num_facilities),
                    "num_clients": int(num_clients),
                    "size": size,
                    "difficulty": difficulty,
                    "alpha": round(alpha, 3),
                    "beta": round(actual_beta, 3),
                    "total_demand": round(total_demand, 2),
                    "total_capacity": round(facility_capacities.sum(), 2),
                    "fixed_costs": fixed_costs.tolist(),
                    "service_costs": service_costs.tolist(),
                    "facility_capacities": facility_capacities.tolist(),
                    "client_demands": client_demands.tolist(),
                    "client_coordinates": clients.tolist(),
                    "facility_coordinates": facilities.tolist(),
                    "num_variables": int(num_vars),
                    "num_constraints": int(num_constraints)
                }

                # Save instance to JSON
                with open(instance_path, "w") as f:
                    json.dump(instance_data, f, indent=2)

                # Store summary information
                instances_info.append({
                    "instance_name": instance_name,
                    "instance_id": instance_id,
                    "size": size,
                    "difficulty": difficulty,
                    "num_facilities": num_facilities,
                    "num_clients": num_clients,
                    "num_variables": num_vars,
                    "num_constraints": num_constraints,
                    "alpha": round(alpha, 3),
                    "beta": round(actual_beta, 3),
                    "total_demand": round(total_demand, 2),
                    "total_capacity": round(facility_capacities.sum(), 2),
                    "capacity_slack_pct": round((actual_beta - 1.0) * 100, 1),
                    "avg_fixed_cost": round(fixed_costs.mean(), 2),
                    "avg_service_cost": round(service_costs.mean(), 2),
                    "avg_demand": round(client_demands.mean(), 2),
                    "avg_capacity": round(facility_capacities.mean(), 2)
                })
                
                # Create visualization with correct coordinate bounds
                plt.figure(figsize=(10, 10))
                
                # Adjust marker sizes based on instance size for visibility
                if num_clients > 500:
                    client_size = 5
                    facility_size = 50
                elif num_clients > 200:
                    client_size = 10
                    facility_size = 80
                else:
                    client_size = 20
                    facility_size = 120
                
                plt.scatter(facilities[:, 0], facilities[:, 1], 
                           c='red', marker='s', s=facility_size, alpha=0.8, 
                           edgecolors='darkred', linewidths=2, 
                           label=f'Facilities (n={num_facilities})')
                plt.scatter(clients[:, 0], clients[:, 1], 
                           c='blue', marker='o', s=client_size, alpha=0.5, 
                           label=f'Clients (n={num_clients})')
                
                plt.title(f"CFLP Instance {instance_id:03d}: {size.capitalize()}-{difficulty.capitalize()}\n"
                         f"Facilities: {num_facilities}, Clients: {num_clients}, α={alpha:.2f}, β={actual_beta:.2f}",
                         fontsize=14, fontweight='bold')
                plt.xlabel("X Coordinate", fontsize=12)
                plt.ylabel("Y Coordinate", fontsize=12)
                
                # Set proper limits based on actual coordinate space (0-1000)
                plt.xlim(-50, 1050)
                plt.ylim(-50, 1050)
                
                plt.legend(loc='upper right', fontsize=10)
                plt.grid(alpha=0.3, linestyle='--')
                plt.tight_layout()
                
                # Save visualization
                viz_path = os.path.join(folder, instance_name.replace(".json", ".png"))
                plt.savefig(viz_path, dpi=150, bbox_inches='tight')
                plt.close()

                instance_id += 1

    # Create summary Excel file
    summary_df = pd.DataFrame(instances_info)
    summary_path = os.path.join(folder, "instances_summary.xlsx")
    
    with pd.ExcelWriter(summary_path, engine='openpyxl') as writer:
        # Full summary
        summary_df.to_excel(writer, sheet_name='All Instances', index=False)
        
        # Statistics by size
        size_stats = summary_df.groupby('size').agg({
            'num_facilities': ['min', 'max', 'mean'],
            'num_clients': ['min', 'max', 'mean'],
            'alpha': ['min', 'max', 'mean'],
            'beta': ['min', 'max', 'mean'],
            'num_variables': ['min', 'max', 'mean'],
            'num_constraints': ['min', 'max', 'mean'],
            'instance_id': 'count'
        }).round(2)
        size_stats.to_excel(writer, sheet_name='Size Statistics')
        
        # Statistics by difficulty
        diff_stats = summary_df.groupby('difficulty').agg({
            'alpha': ['min', 'max', 'mean'],
            'beta': ['min', 'max', 'mean'],
            'total_demand': ['min', 'max', 'mean'],
            'total_capacity': ['min', 'max', 'mean'],
            'instance_id': 'count'
        }).round(2)
        diff_stats.to_excel(writer, sheet_name='Difficulty Statistics')
        
        # Size-Difficulty matrix
        matrix = summary_df.pivot_table(
            values='instance_id', 
            index='size', 
            columns='difficulty', 
            aggfunc='count',
            fill_value=0
        )
        matrix.to_excel(writer, sheet_name='Distribution Matrix')
    
    print(f"\nSuccessfully generated {len(instances_info)} instances in '{folder}/'")
    print(f"   Summary saved to: {summary_path}")
    print(f"\nDistribution Summary:")
    print(summary_df.groupby(['size', 'difficulty']).size().unstack(fill_value=0))


def load_cflp_instance(file_path: str) -> Tuple[int, int, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load a Capacitated Facility Location Problem instance from a JSON file.
    
    Parameters:
    -----------
    file_path : str
        Path to the JSON file containing the instance data
        
    Returns:
    --------
    num_facilities : int
        Number of facilities
    num_clients : int
        Number of clients
    fixed_costs : np.ndarray
        Fixed costs for each facility (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    """
    with open(file_path, 'r') as f:
        instance_data = json.load(f)
    
    num_facilities = instance_data['num_facilities']
    num_clients = instance_data['num_clients']
    fixed_costs = np.array(instance_data['fixed_costs'])
    service_costs = np.array(instance_data['service_costs'])
    facility_capacities = np.array(instance_data['facility_capacities'])
    client_demands = np.array(instance_data['client_demands'])
    
    return num_facilities, num_clients, fixed_costs, service_costs, facility_capacities, client_demands

print("✅ CFLP instance generation module loaded successfully.")

# Uncomment the line below to test the instance generation
# generate_instances(folder="cflp_instances", num_total_instances=144, seed=42)

---

## 4. Solution Encoding (Representation) and Utility Functions

In this section, we define the solution encoding for the Facility Location Problem (FLP) and implement utility functions to evaluate and manipulate solutions. 

The solution is represented as a dictionary with two components:
1. `facilities`: A binary list indicating which facilities are open (1) or closed (0).
2. `assignments`: A list where each element represents the index of the facility assigned to each client.

For example, a dictionary representing a solution might look like this:
{
  "facilities": [1, 0, 1, 0, 1],
  "assignments": [0, 2, 4, 2, 0, 2, 4]
}
- Facilities: [1, 0, 1, 0, 1] (facilities 1, 3, and 5 are open)
- Assignments: [0, 2, 4, 2, 0, 2, 4] (client 1 assigned to facility 1, client 2 to facility 3, client 3 to facility 5, etc.)

The functions related to solution encoding include:
- `compute_total_cost`: Calculate total cost of a solution
- `is_feasible`: Check if a solution is feasible
- `assign_clients_to_nearest_open_facility`: Assign clients to the nearest open facility
- `copy_solution`: Create a deep copy of a solution
- `greedy_initial_solution`: Generate an initial solution using a greedy heuristic. It consists to:
    1. Start with all facilities closed.
    2. Iteratively open the facility that provides the largest cost reduction until no further improvement is possible.
    3. Assign clients to the nearest open facility.

In [ ]:
def compute_total_cost(fixed_costs: np.ndarray, service_costs: np.ndarray, 
                       facility_opened: np.ndarray, client_assignments: np.ndarray) -> float:
    """
    Compute the total cost of a given solution.
    
    Parameters:
    -----------
    fixed_costs : np.ndarray
        Fixed costs for each facility (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
        
    Returns:
    --------
    total_cost : float
        Total cost of the solution
    """
    total_fixed_cost = np.sum(fixed_costs * facility_opened)
    total_service_cost = np.sum(service_costs[np.arange(len(client_assignments)), client_assignments])
    
    total_cost = total_fixed_cost + total_service_cost
    return total_cost

def is_feasible(facility_opened: np.ndarray, client_assignments: np.ndarray,
                facility_capacities: np.ndarray, client_demands: np.ndarray) -> bool:
    """
    Check if a given solution is feasible for CFLP.
    
    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
        
    Returns:
    --------
    feasible : bool
        True if the solution is feasible, False otherwise
    """
    # Check if clients are only assigned to open facilities
    for assigned_facility in client_assignments:
        if facility_opened[assigned_facility] == 0:
            return False
    
    # Check capacity constraints
    for j in range(len(facility_opened)):
        if facility_opened[j] == 1:
            assigned_clients = np.where(client_assignments == j)[0]
            total_demand = client_demands[assigned_clients].sum()
            if total_demand > facility_capacities[j] + 1e-6:  # Small tolerance for floating point
                return False
    
    return True

def assign_clients_to_nearest_open_facility(service_costs: np.ndarray, facility_opened: np.ndarray,
                                            facility_capacities: np.ndarray, client_demands: np.ndarray) -> np.ndarray:
    """
    Assign each client to the nearest opened facility respecting capacity constraints (CFLP).
    Uses a fast capacity-aware greedy heuristic with demand-based sorting.
    
    Parameters:
    -----------
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
        
    Returns:
    --------
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    """
    num_clients = service_costs.shape[0]
    opened_indices = np.where(facility_opened == 1)[0]
    
    if len(opened_indices) == 0:
        raise ValueError("No facilities are open!")
    
    # Initialize assignments and track remaining capacity
    client_assignments = np.full(num_clients, -1, dtype=int)
    remaining_capacity = facility_capacities.copy()
    
    # Get costs only for opened facilities
    costs_to_open = service_costs[:, opened_indices]
    
    # Assign each client to nearest facility with remaining capacity
    # Process clients in natural order (no sorting) for flexibility
    for client_idx in range(num_clients):
        costs = costs_to_open[client_idx]
        sorted_fac_positions = np.argsort(costs)
        
        assigned = False
        for pos in sorted_fac_positions:
            fac_idx = opened_indices[pos]
            if remaining_capacity[fac_idx] >= client_demands[client_idx]:
                client_assignments[client_idx] = fac_idx
                remaining_capacity[fac_idx] -= client_demands[client_idx]
                assigned = True
                break
        
        if not assigned:
            # Infeasible - assign to nearest anyway (will be caught by is_feasible)
            nearest_fac = opened_indices[sorted_fac_positions[0]]
            client_assignments[client_idx] = nearest_fac
    
    return client_assignments

def copy_solution(facility_opened: np.ndarray, client_assignments: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Create a deep copy of a solution.
    
    Parameters:
    -----------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
        
    Returns:
    --------
    facility_opened_copy : np.ndarray
        Copy of the facility opened array
    client_assignments_copy : np.ndarray
        Copy of the client assignments array
    """
    facility_opened_copy = np.copy(facility_opened)
    client_assignments_copy = np.copy(client_assignments)
    
    return facility_opened_copy, client_assignments_copy

def greedy_initial_solution(num_clients: int, num_facilities: int, service_costs: np.ndarray,
                           facility_capacities: np.ndarray, client_demands: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate an initial solution using a greedy heuristic for CFLP.
    
    Parameters:
    -----------
    num_clients : int
        Number of clients
    num_facilities : int
        Number of facilities
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)

    Returns:
    --------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    """
    # Simple greedy approach: open facilities with best average service cost
    # until we have enough capacity to serve all clients
    
    # Calculate average service cost for each facility
    avg_service_cost_per_facility = service_costs.mean(axis=0)
    
    # Sort facilities by average service cost (ascending - lower is better)
    sorted_by_cost = np.argsort(avg_service_cost_per_facility)
    
    # Start with all facilities closed
    facility_opened = np.zeros(num_facilities, dtype=int)
    total_demand = np.sum(client_demands)
    
    # Open facilities in order of lowest average service cost until feasible
    for fac_idx in sorted_by_cost:
        facility_opened[fac_idx] = 1
        
        # Try assigning clients to open facilities
        client_assignments = assign_clients_to_nearest_open_facility(
            service_costs, facility_opened, facility_capacities, client_demands
        )
        
        # Check if feasible
        if is_feasible(facility_opened, client_assignments, facility_capacities, client_demands):
            # We have a feasible solution - stop here!
            break
    
    # If still not feasible (rare case), open all facilities
    if not is_feasible(facility_opened, client_assignments, facility_capacities, client_demands):
        facility_opened = np.ones(num_facilities, dtype=int)
        client_assignments = assign_clients_to_nearest_open_facility(
            service_costs, facility_opened, facility_capacities, client_demands
        )
    
    return facility_opened, client_assignments

print("✅ CFLP utility functions module loaded successfully.")

# Example usage: Uncomment the the following lines to load an instance and test utility functions
# file_path = "cflp_instances/001_cflp_small_easy_fac15_cli50.json"
# num_facilities, num_clients, fixed_costs, service_costs, facility_capacities, client_demands = load_cflp_instance(file_path)
# # combine to a dictionary
# instance = {
#     "num_facilities": num_facilities,
#     "num_clients": num_clients,
#     "fixed_costs": fixed_costs,
#     "service_costs": service_costs,
#     "facility_capacities": facility_capacities,
#     "client_demands": client_demands
# }
# # Generate initial solution
# facility_opened, client_assignments = greedy_initial_solution(
#     num_clients=instance["num_clients"], 
#     num_facilities=instance["num_facilities"], 
#     service_costs=instance["service_costs"],
#     facility_capacities=instance["facility_capacities"],
#     client_demands=instance["client_demands"]
# )
# # Display solution summary (just the first and last ten assignments)
# print(f"Facility opened: {facility_opened}")
# print(f"Client assignments: {client_assignments[:10]} ... {client_assignments[-10:]}")

# # Test total cost computation
# total_cost = compute_total_cost(
#     fixed_costs=instance["fixed_costs"], 
#     service_costs=instance["service_costs"], 
#     facility_opened=facility_opened, 
#     client_assignments=client_assignments
# )
# print(f"Total cost of the initial solution: {total_cost}")

# # Test feasibility check
# feasible = is_feasible(
#     facility_opened=facility_opened, 
#     client_assignments=client_assignments,
#     facility_capacities=instance["facility_capacities"],
#     client_demands=instance["client_demands"]
# )
# print(f"Is the initial solution feasible? {'Yes' if feasible else 'No'}")


---

## 5. Variable Neighborhood Search (VNS) Implementation

In this section, we implement the Variable Neighborhood Search (VNS) algorithm to solve the Capacitated Facility Location Problem (CFLP). VNS is a metaheuristic that systematically changes the neighborhood structures within the search process to escape local optima and explore the solution space more effectively.

The VNS algorithm consists of the following key components:
1. **Neighborhood Structures**: Define different ways to modify the current solution, such as opening/closing facilities or reassigning clients.
2. **Shaking**: Randomly perturb the current solution to explore a new neighborhood.
3. **Local Search**: Apply a local search method to find a local optimum within the current neighborhood.
4. **Acceptance Criterion**: Decide whether to accept the new solution based on its quality compared to the current solution. Accept the new solution if it is better than the current one.

### 5.1 Neighborhood Structures

We define multiple neighborhood structures to explore different aspects of the solution space:

a) **N1**: Open a closed facility, function `n1_open_facility`.

- The function randomly selects a closed facility and opens it, then reassigns clients to the nearest open facility.
- The reassignment is done based on the minimum distance to the open facilities.
- The function returns a list of a certain number `max_neighbors` of generated neighbor solutions. If `max_neighbors` is set to 0, it returns all possible neighbors. The parameter `max_neighbors` controls the maximum number of neighbors to generate.
- The function also ensures that the generated neighbors are unique and feasible.

b) **N2**: Close an open facility, function `n2_close_open_facility`.

- The function randomly selects an open facility and closes it, then randomly selects a closed facility and opens it, then reassigns clients to the nearest open facility.
- The reassignment is done based on the minimum distance to the open facilities.
- The function returns a list of a certain number `max_neighbors` of generated neighbor solutions. If `max_neighbors` is set to 0, it returns all possible neighbors. The parameter `max_neighbors` controls the maximum number of neighbors to generate.
- The function also ensures that the generated neighbors are unique and feasible.

c) **N3**: Reassign a client to a different open facility, function `n3_reassign_client`.

- The function randomly selects a client and reassigns it to a different open facility.
- The function returns a list of a certain number `max_neighbors` of generated neighbor solutions. If `max_neighbors` is set to 0, it returns all possible neighbors. The parameter `max_neighbors` controls the maximum number of neighbors to generate.
- The function also ensures that the generated neighbors are unique and feasible.

d) **N4**: Swap the assignments of two clients, function `n4_swap_clients`.

- The function randomly selects two clients and swaps their assignments to open facilities.
- The function returns a list of a certain number `max_neighbors` of generated neighbor solutions. If `max_neighbors` is set to 0, it returns all possible neighbors. The parameter `max_neighbors` controls the maximum number of neighbors to generate.
- The function also ensures that the generated neighbors are unique and feasible.

In [ ]:
def n1_open_facility(max_neighbors: int, facility_opened: np.ndarray, service_costs: np.ndarray,
                     facility_capacities: np.ndarray, client_demands: np.ndarray) -> List[np.ndarray]:
    """
    Generate neighboring solutions by opening closed facilities (CFLP version).
    
    Parameters:
    -----------
    max_neighbors : int
        Maximum number of neighbors to generate
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
        
    Returns:
    --------
    neighbors : List[np.ndarray]
        List of neighboring facility opened arrays and their corresponding client assignments
    """
    neighbors = []
    closed_indices = np.where(facility_opened == 0)[0]
    np.random.shuffle(closed_indices)
    
    for j in closed_indices:
        if len(neighbors) >= max_neighbors:
            break
        neighbor_facility_opened = facility_opened.copy()
        neighbor_facility_opened[j] = 1
        neighbor_client_assignments = assign_clients_to_nearest_open_facility(
            service_costs, neighbor_facility_opened, facility_capacities, client_demands
        )
        
        # Check feasibility with capacity constraints
        if is_feasible(neighbor_facility_opened, neighbor_client_assignments, facility_capacities, client_demands):
            neighbors.append((neighbor_facility_opened, neighbor_client_assignments))

    return neighbors

def n2_close_open_facility(max_neighbors: int, facility_opened: np.ndarray, service_costs: np.ndarray,
                           facility_capacities: np.ndarray, client_demands: np.ndarray) -> List[np.ndarray]:
    """
    Generate neighboring solutions by CLOSING opened facilities (CFLP version).
    This helps reduce fixed costs by trying to close unnecessary facilities.
    
    Parameters:
    -----------
    max_neighbors : int
        Maximum number of neighbors to generate
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
        
    Returns:
    --------
    neighbors : List[np.ndarray]
        List of neighboring facility opened arrays and their corresponding client assignments
    """
    neighbors = []
    opened_indices = np.where(facility_opened == 1)[0]
    closed_indices = np.where(facility_opened == 0)[0]
    
    # Need at least 1 open and 1 closed facility for swap
    if len(opened_indices) == 0 or len(closed_indices) == 0:
        return neighbors
    
    np.random.shuffle(opened_indices)
    np.random.shuffle(closed_indices)
    
    # Swap: close one open facility and open one closed facility
    count = 0
    for j_open in opened_indices:
        if count >= max_neighbors:
            break
        for j_close in closed_indices[:3]:  # Try only 3 closed facilities per open one
            if count >= max_neighbors:
                break
            
            neighbor_facility_opened = facility_opened.copy()
            neighbor_facility_opened[j_open] = 0  # Close this one
            neighbor_facility_opened[j_close] = 1  # Open this one
            
            # Ensure feasibility by reassigning clients with capacity constraints
            neighbor_client_assignments = assign_clients_to_nearest_open_facility(
                service_costs, neighbor_facility_opened, facility_capacities, client_demands
            )
            
            # Only add if still feasible
            if is_feasible(neighbor_facility_opened, neighbor_client_assignments, facility_capacities, client_demands):
                neighbors.append((neighbor_facility_opened, neighbor_client_assignments))
                count += 1

    return neighbors

def n3_swap_facilities(max_neighbors: int, facility_opened: np.ndarray, service_costs: np.ndarray,
                       facility_capacities: np.ndarray, client_demands: np.ndarray) -> List[np.ndarray]:
    """
    Generate neighboring solutions by swapping opened and closed facilities (CFLP version).
    
    Parameters:
    -----------
    max_neighbors : int
        Maximum number of neighbors to generate
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
        
    Returns:
    --------
    neighbors : List[np.ndarray]
        List of neighboring facility opened arrays and their corresponding client assignments
    """
    neighbors = []
    opened_indices = np.where(facility_opened == 1)[0]
    closed_indices = np.where(facility_opened == 0)[0]
    
    if len(opened_indices) == 0 or len(closed_indices) == 0:
        return neighbors
    
    np.random.shuffle(opened_indices)
    np.random.shuffle(closed_indices)
    
    # Simple 1-to-1 swaps work best
    for j_open in opened_indices:
        for j_close in closed_indices:
            if len(neighbors) >= max_neighbors:
                break
            neighbor_facility_opened = facility_opened.copy()
            neighbor_facility_opened[j_open] = 0
            neighbor_facility_opened[j_close] = 1
            # Ensure feasibility by reassigning clients with capacity constraints
            neighbor_client_assignments = assign_clients_to_nearest_open_facility(
                service_costs, neighbor_facility_opened, facility_capacities, client_demands
            )
            if is_feasible(neighbor_facility_opened, neighbor_client_assignments, facility_capacities, client_demands):
                neighbors.append((neighbor_facility_opened, neighbor_client_assignments))
        if len(neighbors) >= max_neighbors:
            break

    return neighbors

def n4_reassign_clients(max_neighbors: int, facility_opened: np.ndarray, client_assignments: np.ndarray, 
                        service_costs: np.ndarray, facility_capacities: np.ndarray, 
                        client_demands: np.ndarray) -> List[np.ndarray]:
    """
    Generate neighboring solutions by reassigning clients to different opened facilities (CFLP version).
    
    Parameters:
    -----------
    max_neighbors : int
        Maximum number of neighbors to generate
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
        
    Returns:
    --------
    neighbors : List[np.ndarray]
        List of neighboring facility opened arrays and their corresponding client assignments
    """
    neighbors = []
    num_clients = len(client_assignments)
    opened_indices = np.where(facility_opened == 1)[0]
    
    # Randomly shuffle clients to try different reassignments
    client_indices = list(range(num_clients))
    np.random.shuffle(client_indices)
    
    for i in client_indices:
        if len(neighbors) >= max_neighbors:
            break
            
        current_facility = client_assignments[i]
        alternative_facilities = opened_indices[opened_indices != current_facility]
        np.random.shuffle(alternative_facilities)
        
        # Try reassigning to alternative facilities
        for j in alternative_facilities:
            if len(neighbors) >= max_neighbors:
                break
            neighbor_client_assignments = client_assignments.copy()
            neighbor_client_assignments[i] = j
            
            # Check capacity feasibility before adding as neighbor
            if is_feasible(facility_opened.copy(), neighbor_client_assignments, facility_capacities, client_demands):
                neighbors.append((facility_opened.copy(), neighbor_client_assignments))
                break  # Only add one neighbor per client to avoid too many similar neighbors

    return neighbors

print("✅ Neighborhood generation modules loaded successfully.")

### 5.2 Shaking function

The shaking function is responsible for perturbing the current solution to explore a new neighborhood. It randomly selects one of the defined neighborhood structures and applies it to the current solution to generate a new solution.

In [ ]:
def shaking(current_solution: Tuple[np.ndarray, np.ndarray], max_neighbors: int, 
             service_costs: np.ndarray, facility_capacities: np.ndarray, 
             client_demands: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Perform shaking by randomly selecting a neighborhood structure and generating a neighbor solution (CFLP version).
    
    Parameters:
    -----------
    current_solution : Tuple[np.ndarray, np.ndarray]
        Current solution as a tuple of (facility_opened, client_assignments)
    max_neighbors : int
        Maximum number of neighbors to generate in the selected neighborhood
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
        
    Returns:
    --------
    new_solution : Tuple[np.ndarray, np.ndarray]
        New solution after shaking
    """
    facility_opened, client_assignments = current_solution
    neighborhood_functions = [n1_open_facility, n2_close_open_facility, n3_swap_facilities, n4_reassign_clients]
    
    # Randomly select a neighborhood structure
    neighborhood_func = np.random.choice(neighborhood_functions)
    
    # Generate neighbors using the selected neighborhood structure
    if neighborhood_func == n1_open_facility:
        neighbors = neighborhood_func(max_neighbors, facility_opened, service_costs, facility_capacities, client_demands)
    elif neighborhood_func == n2_close_open_facility:
        neighbors = neighborhood_func(max_neighbors, facility_opened, service_costs, facility_capacities, client_demands)
    elif neighborhood_func == n3_swap_facilities:
        neighbors = neighborhood_func(max_neighbors, facility_opened, service_costs, facility_capacities, client_demands)
    elif neighborhood_func == n4_reassign_clients:
        neighbors = neighborhood_func(max_neighbors, facility_opened, client_assignments, service_costs, 
                                     facility_capacities, client_demands)
    
    # Randomly select one neighbor as the new solution
    if neighbors:
        new_solution = neighbors[np.random.randint(len(neighbors))]
        return new_solution
    else:
        return current_solution

print("✅ Shaking module loaded successfully.")

### 5.3 Local Search

The local search function explores the neighborhood of the current solution to find a local optimum. It iteratively applies the neighborhood structures to generate neighboring solutions and evaluates their costs. If a neighboring solution has a lower cost than the current solution, it becomes the new current solution. This process continues until no further improvements can be found in the neighborhood. The parameter `neighborhood_structures` allows the user to specify which neighborhood structures to use during the local search.

In [ ]:
def local_search(current_solution: Tuple[np.ndarray, np.ndarray], 
                 service_costs: np.ndarray, 
                 fixed_costs: np.ndarray,
                 facility_capacities: np.ndarray,
                 client_demands: np.ndarray,
                 neighborhood_structures: str,
                 max_neighbors: int = 10) -> Tuple[np.ndarray, np.ndarray]:
    """
    Perform local search to find a local optimum solution (CFLP version).
    
    Parameters:
    -----------
    current_solution : Tuple[np.ndarray, np.ndarray]
        Current solution as a tuple of (facility_opened, client_assignments)
    service_costs : np.ndarray
        Service costs matrix (shape: num_clients × num_facilities)
    fixed_costs : np.ndarray
        Fixed costs for each facility (shape: num_facilities)
    facility_capacities : np.ndarray
        Capacity of each facility (shape: num_facilities)
    client_demands : np.ndarray
        Demand of each client (shape: num_clients)
    neighborhood_structures : str
        List of neighborhood structure to use during local search
    max_neighbors : int
        Maximum number of neighbors to generate in each neighborhood structure
        
    Returns:
    --------
    best_solution : Tuple[np.ndarray, np.ndarray]
        Best solution found after local search
    """
    improved = True
    best_solution = current_solution
    best_cost = compute_total_cost(
        fixed_costs=fixed_costs, 
        service_costs=service_costs, 
        facility_opened=best_solution[0], 
        client_assignments=best_solution[1]
    )
    while improved:
        improved = False
        for neighborhood in neighborhood_structures:
            if neighborhood == 'n1':
                neighbors = n1_open_facility(max_neighbors, best_solution[0], service_costs, 
                                           facility_capacities, client_demands)
            elif neighborhood == 'n2':
                neighbors = n2_close_open_facility(max_neighbors, best_solution[0], service_costs,
                                                  facility_capacities, client_demands)
            elif neighborhood == 'n3':
                neighbors = n3_swap_facilities(max_neighbors, best_solution[0], service_costs,
                                              facility_capacities, client_demands)
            elif neighborhood == 'n4':
                neighbors = n4_reassign_clients(max_neighbors, best_solution[0], best_solution[1], service_costs,
                                               facility_capacities, client_demands)
            else:
                continue
            
            for neighbor in neighbors:
                neighbor_cost = compute_total_cost(
                    fixed_costs=fixed_costs, 
                    service_costs=service_costs, 
                    facility_opened=neighbor[0], 
                    client_assignments=neighbor[1]
                )
                if neighbor_cost < best_cost:
                    best_solution = neighbor
                    best_cost = neighbor_cost
                    improved = True
                    break
            if improved:
                break
    return best_solution

print("✅ Local Search module loaded successfully.")

# Test Local Search (uncomment the following lines to run a test)
# current_solution = greedy_initial_solution(
#     num_clients=instance["num_clients"], 
#     num_facilities=instance["num_facilities"], 
#     service_costs=instance["service_costs"]
# )
# print("Total cost before local search:", compute_total_cost(
#     fixed_costs=instance["fixed_costs"], 
#     service_costs=instance["service_costs"], 
#     facility_opened=current_solution[0], 
#     client_assignments=current_solution[1]
# ))
# best_solution = local_search(
#     current_solution=current_solution,
#     service_costs=instance["service_costs"],
#     fixed_costs=instance["fixed_costs"],
#     neighborhood_structures=['n1', 'n2', 'n3', 'n4'],
#     max_neighbors=10
# )
# best_facility_opened, best_client_assignments = best_solution
# best_total_cost = compute_total_cost(
#     fixed_costs=instance["fixed_costs"], 
#     service_costs=instance["service_costs"], 
#     facility_opened=best_facility_opened, 
#     client_assignments=best_client_assignments
# )
# print(f"Total cost after local search: {best_total_cost}")


---

### 5.4 Variable Neighborhood Search (VNS) - MAIN ALGORITHM

The main VNS algorithm orchestrates the overall search process. It starts with an initial solution and iteratively applies the shaking and local search functions to explore the solution space. The algorithm continues until a stopping criterion is met, such as reaching a maximum number of iterations or a time limit. The best solution found during the search is returned as the final output.

The VNS algorithm can be summarized in the following steps:
1. Generate an initial solution using a greedy heuristic.
2. Define a set of neighborhood structures.
3. Iteratively perform the following until a stopping criterion is met:
   a. Apply the shaking function to generate a new solution.
   b. Apply the local search function to the new solution.
   c. If the new solution is better than the current solution, update the current solution.

The algorithm returns the best solution found along with its cost and a track of costs over iterations for analysis.

**⚠️ IMPORTANT NOTE FOR STUDENTS:**
This VNS implementation contains a subtle bug that affects its performance on many instances. You may observe solution gaps greater than 20% on multiple instances. This is **intentional for educational purposes**. 

For the bonus question (3 marks), you can identify and fix this bug to significantly improve VNS performance. See the Student Guide (Section 8) for details on the bonus challenge.

In [ ]:
def vns_solve_cflp(instance: dict, max_iterations: int = 100, max_no_improve: int = 20, 
            max_neighbors: int = 10, neighborhood_structures: List[str] = ['n1', 'n2', 'n3', 'n4']) -> Tuple[Tuple[np.ndarray, np.ndarray], List[float]]:
    """
    Variable Neighborhood Search for the Capacitated Facility Location Problem (CFLP)
    
    Parameters:
    -----------
    instance : dict
        Dictionary containing the CFLP instance data with keys:
            - num_facilities
            - num_clients
            - fixed_costs
            - service_costs
            - facility_capacities
            - client_demands
    max_iterations : int
        Maximum number of VNS iterations
    max_no_improve : int
        Maximum number of iterations without improvement before stopping
    max_neighbors : int
        Maximum number of neighbors to generate in each neighborhood structure
    neighborhood_structures : List[str]
        List of neighborhood structures to use during local search
        Options are: 'n1', 'n2', 'n3', 'n4'
        Default is all available structures
    
    Returns:
    --------
    best_solution : Tuple[np.ndarray, np.ndarray]
        Best solution found as a tuple of (facility_opened, client_assignments)
    trajectory_data : Tuple[List[float], List[int]]
        Tuple containing:
        - trajectory: List of best costs found at improvement iterations
        - trajectory_iterations: List of actual iteration numbers where improvements occurred
    """
    # Generate initial solution
    current_solution = greedy_initial_solution(
        num_clients=instance["num_clients"], 
        num_facilities=instance["num_facilities"], 
        service_costs=instance["service_costs"],
        facility_capacities=instance["facility_capacities"],
        client_demands=instance["client_demands"]
    )
    
    # CRITICAL: Apply local search to improve the initial solution!
    # This ensures we start from a local optimum, not just a greedy solution
    # print("Improving initial solution with local search...")
    current_solution = local_search(
        current_solution=current_solution,
        service_costs=instance["service_costs"],
        fixed_costs=instance["fixed_costs"],
        facility_capacities=instance["facility_capacities"],
        client_demands=instance["client_demands"],
        neighborhood_structures=neighborhood_structures,
        max_neighbors=max_neighbors
    )
    
    current_cost = compute_total_cost(
        fixed_costs=instance["fixed_costs"], 
        service_costs=instance["service_costs"], 
        facility_opened=current_solution[0], 
        client_assignments=current_solution[1]
    )
    best_solution = current_solution
    best_cost = current_cost
    trajectory = [best_cost]
    trajectory_iterations = [0]  # Track actual iteration numbers for plotting
    no_improve_count = 0 
    iteration = 0
    
    # print(f"Initial solution cost after local search: {best_cost:.2f}")
    print("Starting VNS for CFLP...")
    while iteration < max_iterations and no_improve_count < max_no_improve:
        iteration += 1  # Increment at start so iterations are 1-indexed
        
        # Shaking - FIXED: Always shake from best solution, not current
        new_solution = shaking(best_solution, max_neighbors, instance["service_costs"],
                              instance["facility_capacities"], instance["client_demands"])
        
        # Local Search
        new_solution = local_search(
            current_solution=new_solution,
            service_costs=instance["service_costs"],
            fixed_costs=instance["fixed_costs"],
            facility_capacities=instance["facility_capacities"],
            client_demands=instance["client_demands"],
            neighborhood_structures=neighborhood_structures,
            max_neighbors=max_neighbors
        )
        
        new_cost = compute_total_cost(
            fixed_costs=instance["fixed_costs"], 
            service_costs=instance["service_costs"], 
            facility_opened=new_solution[0], 
            client_assignments=new_solution[1]
        )
        
        # Acceptance Criterion - FIXED: Only accept improving solutions
        if new_cost < best_cost:
            best_solution = new_solution
            best_cost = new_cost
            current_solution = new_solution  # Update current to new best
            current_cost = new_cost
            trajectory.append(best_cost)
            trajectory_iterations.append(iteration)
            no_improve_count = 0
            # print(f"Iteration {iteration}: New best cost found: {best_cost:.2f}")
        else:
            no_improve_count += 1
            # DO NOT accept worse solution - keep shaking from best solution
            # This prevents cycling between same local optima
    
    print(f"VNS completed after {iteration} iterations.")
    print(f"Best cost: {best_cost:.2f}")
    
    return best_solution, (trajectory, trajectory_iterations)

print("✅ VNS module loaded successfully.")

# Test VNS (uncomment the following lines to run a test)
# best_solution, trajectory = vns_solve_cflp(
#     instance=instance, 
#     max_iterations=100, 
#     max_no_improve=20, 
#     max_neighbors=10, 
#     neighborhood_structures=['n1', 'n2', 'n3', 'n4']
# )
# print(f"Final solution cost: {trajectory[0][-1]:.2f}")
# print(f"Convergence trajectory: {trajectory}")

---

## 6. Exact Solution using Gurobi

In this section, we implement the exact solution for the Capacitated Facility Location Problem (CFLP) using the Gurobi optimization solver. Gurobi is a powerful tool for solving mixed-integer programming (MIP) problems, and it can provide optimal solutions for small to medium-sized instances of the CFLP.


In [ ]:
def gurobi_solve_cflp(instance: dict, time_limit: int = 300, check_size_limit: bool = False) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Diagnostic function to understand why VNS is not improving.
    Tests each component and prints detailed information.
    
    Parameters:
    -----------
    instance : dict
        CFLP instance to diagnose
    max_iterations : int
        Number of iterations to test (default: 10 for quick diagnosis)
    max_neighbors : int
        Max neighbors per neighborhood
    """
    print("="*80)
    print("VNS DIAGNOSTIC TOOL")
    print("="*80)
    
    # Step 1: Test initial solution
    print("\n1. TESTING INITIAL SOLUTION GENERATION:")
    print("-"*80)
    initial_solution = greedy_initial_solution(
        num_clients=instance["num_clients"],
        num_facilities=instance["num_facilities"],
        service_costs=instance["service_costs"],
        facility_capacities=instance["facility_capacities"],
        client_demands=instance["client_demands"]
    )
    initial_cost = compute_total_cost(
        fixed_costs=instance["fixed_costs"],
        service_costs=instance["service_costs"],
        facility_opened=initial_solution[0],
        client_assignments=initial_solution[1]
    )
    num_open_facilities = np.sum(initial_solution[0])
    print(f"   Initial cost: {initial_cost:.2f}")
    print(f"   Facilities opened: {num_open_facilities}/{instance['num_facilities']}")
    print(f"   Is feasible: {is_feasible(initial_solution[0], initial_solution[1], instance['facility_capacities'], instance['client_demands'])}")
    
    # Step 2: Test local search on initial solution
    print("\n2. TESTING LOCAL SEARCH ON INITIAL SOLUTION:")
    print("-"*80)
    improved_solution = local_search(
        current_solution=initial_solution,
        service_costs=instance["service_costs"],
        fixed_costs=instance["fixed_costs"],
        facility_capacities=instance["facility_capacities"],
        client_demands=instance["client_demands"],
        neighborhood_structures=['n1', 'n2', 'n3', 'n4'],
        max_neighbors=max_neighbors
    )
    improved_cost = compute_total_cost(
        fixed_costs=instance["fixed_costs"],
        service_costs=instance["service_costs"],
        facility_opened=improved_solution[0],
        client_assignments=improved_solution[1]
    )
    improvement = initial_cost - improved_cost
    print(f"   Cost after local search: {improved_cost:.2f}")
    print(f"   Improvement: {improvement:.2f} ({(improvement/initial_cost*100):.2f}%)")
    print(f"   Facilities opened: {np.sum(improved_solution[0])}/{instance['num_facilities']}")
    
    # Step 3: Test each neighborhood structure
    print("\n3. TESTING EACH NEIGHBORHOOD STRUCTURE:")
    print("-"*80)
    for neighborhood_name in ['n1', 'n2', 'n3', 'n4']:
        print(f"\n   Testing {neighborhood_name.upper()}:")
        if neighborhood_name == 'n1':
            neighbors = n1_open_facility(max_neighbors, improved_solution[0], instance["service_costs"],
                                       instance["facility_capacities"], instance["client_demands"])
        elif neighborhood_name == 'n2':
            neighbors = n2_close_open_facility(max_neighbors, improved_solution[0], instance["service_costs"],
                                             instance["facility_capacities"], instance["client_demands"])
        elif neighborhood_name == 'n3':
            neighbors = n3_swap_facilities(max_neighbors, improved_solution[0], instance["service_costs"],
                                         instance["facility_capacities"], instance["client_demands"])
        elif neighborhood_name == 'n4':
            neighbors = n4_reassign_clients(max_neighbors, improved_solution[0], improved_solution[1],
                                          instance["service_costs"], instance["facility_capacities"],
                                          instance["client_demands"])
        
        print(f"      Neighbors generated: {len(neighbors)}")
        
        if len(neighbors) > 0:
            # Check costs of neighbors
            neighbor_costs = []
            for neighbor in neighbors:
                cost = compute_total_cost(
                    fixed_costs=instance["fixed_costs"],
                    service_costs=instance["service_costs"],
                    facility_opened=neighbor[0],
                    client_assignments=neighbor[1]
                )
                neighbor_costs.append(cost)
            
            best_neighbor_cost = min(neighbor_costs)
            improvements_found = sum(1 for c in neighbor_costs if c < improved_cost)
            
            print(f"      Best neighbor cost: {best_neighbor_cost:.2f}")
            print(f"      Current cost: {improved_cost:.2f}")
            print(f"      Improvements found: {improvements_found}/{len(neighbors)}")
            if improvements_found > 0:
                print(f"      ✓ This neighborhood CAN find improvements!")
            else:
                print(f"      ✗ This neighborhood found NO improvements")
        else:
            print(f"      ✗ NO neighbors generated (empty neighborhood!)")
    
    # Step 4: Test shaking
    print("\n4. TESTING SHAKING FUNCTION:")
    print("-"*80)
    shaking_costs = []
    for i in range(5):
        shaken_solution = shaking(improved_solution, max_neighbors, instance["service_costs"],
                                 instance["facility_capacities"], instance["client_demands"])
        shaken_cost = compute_total_cost(
            fixed_costs=instance["fixed_costs"],
            service_costs=instance["service_costs"],
            facility_opened=shaken_solution[0],
            client_assignments=shaken_solution[1]
        )
        shaking_costs.append(shaken_cost)
    
    print(f"   Original cost: {improved_cost:.2f}")
    print(f"   Shaken costs (5 trials): {[f'{c:.2f}' for c in shaking_costs]}")
    print(f"   Average shaken cost: {np.mean(shaking_costs):.2f}")
    print(f"   Best shaken cost: {min(shaking_costs):.2f}")
    print(f"   Worst shaken cost: {max(shaking_costs):.2f}")
    
    # Step 5: Run a few VNS iterations with detailed tracking
    print("\n5. RUNNING VNS ITERATIONS WITH DETAILED TRACKING:")
    print("-"*80)
    current_solution = improved_solution
    current_cost = improved_cost
    best_cost = improved_cost
    
    for iteration in range(1, max_iterations + 1):
        # Shaking
        new_solution = shaking(current_solution, max_neighbors, instance["service_costs"],
                              instance["facility_capacities"], instance["client_demands"])
        after_shaking_cost = compute_total_cost(
            fixed_costs=instance["fixed_costs"],
            service_costs=instance["service_costs"],
            facility_opened=new_solution[0],
            client_assignments=new_solution[1]
        )
        
        # Local Search
        new_solution = local_search(
            current_solution=new_solution,
            service_costs=instance["service_costs"],
            fixed_costs=instance["fixed_costs"],
            facility_capacities=instance["facility_capacities"],
            client_demands=instance["client_demands"],
            neighborhood_structures=['n1', 'n2', 'n3', 'n4'],
            max_neighbors=max_neighbors
        )
        after_ls_cost = compute_total_cost(
            fixed_costs=instance["fixed_costs"],
            service_costs=instance["service_costs"],
            facility_opened=new_solution[0],
            client_assignments=new_solution[1]
        )
        
        ls_improvement = after_shaking_cost - after_ls_cost
        
        print(f"\n   Iteration {iteration}:")
        print(f"      After shaking: {after_shaking_cost:.2f}")
        print(f"      After local search: {after_ls_cost:.2f} (LS improved by {ls_improvement:.2f})")
        print(f"      Best so far: {best_cost:.2f}")
        
        if after_ls_cost < best_cost:
            improvement = best_cost - after_ls_cost
            best_cost = after_ls_cost
            current_solution = new_solution
            current_cost = after_ls_cost
            print(f"      ✓✓✓ NEW BEST! Improved by {improvement:.2f}")
        else:
            current_solution = new_solution
            current_cost = after_ls_cost
            print(f"      ✗ No improvement")
    
    print("\n" + "="*80)
    print("DIAGNOSIS SUMMARY:")
    print("="*80)
    print(f"Initial cost: {initial_cost:.2f}")
    print(f"After initial local search: {improved_cost:.2f}")
    print(f"Best cost after {max_iterations} VNS iterations: {best_cost:.2f}")
    print(f"Total improvement: {initial_cost - best_cost:.2f} ({((initial_cost - best_cost)/initial_cost*100):.2f}%)")
    print("="*80)

print("✅ VNS diagnostic tool loaded successfully.")
print("   To use: diagnose_vns_problem(instance, max_iterations=10)")

---

## 6. Exact Solution using Gurobi

In this section, we implement the exact solution for the Capacitated Facility Location Problem (CFLP) using the Gurobi optimization solver. Gurobi is a powerful tool for solving mixed-integer programming (MIP) problems, and it can provide optimal solutions for small to medium-sized instances of the CFLP.


In [ ]:
def gurobi_solve_cflp(instance: dict, time_limit: int = 300, check_size_limit: bool = False) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Solve the Capacitated Facility Location Problem (CFLP) using Gurobi optimization solver.
    
    Parameters:
    -----------
    instance : dict
        Dictionary containing the CFLP instance data with keys:
            - num_facilities
            - num_clients
            - fixed_costs
            - service_costs
            - facility_capacities
            - client_demands
    time_limit : int
        Time limit for the solver in seconds (default: 300 seconds)
    check_size_limit : bool
        If True, check if instance exceeds free Gurobi limits (2000 vars, 2000 constraints)
        and skip solving if it does (default: False for academic/unlimited version)
        
    Returns:
    --------
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened (shape: num_facilities)
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client (shape: num_clients)
    optimal_cost : float
        Optimal total cost of the solution
        
    Raises:
    -------
    ValueError
        If instance exceeds free Gurobi limits (when check_size_limit=True)
    """
    from gurobipy import Model, GRB, quicksum

    num_facilities = instance["num_facilities"]
    num_clients = instance["num_clients"]
    fixed_costs = instance["fixed_costs"]
    service_costs = instance["service_costs"]
    facility_capacities = instance["facility_capacities"]
    client_demands = instance["client_demands"]

    # Check size limits for free Gurobi version
    if check_size_limit:
        num_variables = num_facilities + (num_clients * num_facilities)  # y + x variables
        num_constraints = num_clients + (num_clients * num_facilities) + num_facilities  # assignment + linking + capacity
        
        if num_variables > 2000 or num_constraints > 2000:
            raise ValueError(
                f"Instance exceeds free Gurobi limits: "
                f"{num_variables} variables (max 2000), "
                f"{num_constraints} constraints (max 2000). "
                f"Skipping this instance."
            )

    # Create Gurobi model
    model = Model("Capacitated_Facility_Location_Problem")
    model.setParam('OutputFlag', 0)  # Suppress output
    model.setParam('TimeLimit', time_limit)
    
    # Decision variables
    y = model.addVars(num_facilities, vtype=GRB.BINARY, name="y")  # Facility open variables
    x = model.addVars(num_clients, num_facilities, vtype=GRB.BINARY, name="x")  # Client assignment variables
    
    # Objective function: Minimize total cost
    model.setObjective(
        quicksum(fixed_costs[j] * y[j] for j in range(num_facilities)) +
        quicksum(service_costs[i][j] * x[i, j] for i in range(num_clients) for j in range(num_facilities)),
        GRB.MINIMIZE
    )
    
    # Constraints: Each client must be assigned to exactly one facility
    for i in range(num_clients):
        model.addConstr(quicksum(x[i, j] for j in range(num_facilities)) == 1, name=f"Client_{i}_assignment")
    
    # Constraints: Clients can only be assigned to opened facilities
    for i in range(num_clients):
        for j in range(num_facilities):
            model.addConstr(x[i, j] <= y[j], name=f"Client_{i}_to_Facility_{j}_open")
    
    # NEW: Capacity constraints for CFLP
    for j in range(num_facilities):
        model.addConstr(
            quicksum(client_demands[i] * x[i, j] for i in range(num_clients)) <= facility_capacities[j] * y[j],
            name=f"Facility_{j}_capacity"
        )
    
    # Optimize
    model.optimize()

    # Check if solution was found
    if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT:
        # Extract solution
        facility_opened = np.array([int(round(y[j].X)) for j in range(num_facilities)])
        client_assignments = np.array([int(round(sum(j * x[i, j].X for j in range(num_facilities)))) 
                                        for i in range(num_clients)])
        optimal_cost = model.objVal
        
        return facility_opened, client_assignments, optimal_cost
    else:
        raise Exception(f"Gurobi optimization failed with status: {model.status}")

print("✅ Gurobi module loaded successfully.")

# Test Gurobi (uncomment the following lines to run a test)
# gurobi_facilities, gurobi_assignments, gurobi_cost = gurobi_solve_cflp(instance, time_limit=60)
# print(f"Gurobi optimal cost: {gurobi_cost:.2f}")
# print(f"Facilities opened: {gurobi_facilities}")
# print(f"Number of facilities opened: {gurobi_facilities.sum()}")

In [ ]:
def plot_solution(instance: dict, facility_opened: np.ndarray, 
                  client_assignments: np.ndarray, title: str = "FLP Solution"):
    """
    Visualize a FLP solution showing facilities, clients, and assignments.
    
    Parameters:
    -----------
    instance : dict
        Dictionary containing the FLP instance data
    facility_opened : np.ndarray
        Binary array indicating which facilities are opened
    client_assignments : np.ndarray
        Array indicating the assigned facility for each client
    title : str
        Title for the plot
    """
    # Load coordinate data if available
    with open(file_path, 'r') as f:
        instance_data = json.load(f)
    
    if 'facility_coordinates' not in instance_data or 'client_coordinates' not in instance_data:
        print("Coordinates not available for visualization")
        return
    
    facility_coords = np.array(instance_data['facility_coordinates'])
    client_coords = np.array(instance_data['client_coordinates'])
    
    plt.figure(figsize=(12, 8))
    
    # Plot closed facilities
    closed_facilities = facility_coords[facility_opened == 0]
    if len(closed_facilities) > 0:
        plt.scatter(closed_facilities[:, 0], closed_facilities[:, 1], 
                   c='gray', marker='s', s=150, alpha=0.3, 
                   edgecolors='darkgray', linewidths=2, label='Closed Facilities')
    
    # Plot open facilities
    open_facilities = facility_coords[facility_opened == 1]
    if len(open_facilities) > 0:
        plt.scatter(open_facilities[:, 0], open_facilities[:, 1], 
                   c='red', marker='s', s=200, alpha=0.8, 
                   edgecolors='darkred', linewidths=2, label='Open Facilities')
    
    # Plot clients and assignments
    colors = plt.cm.tab20(np.linspace(0, 1, len(open_facilities)))
    open_indices = np.where(facility_opened == 1)[0]
    
    for idx, facility_idx in enumerate(open_indices):
        assigned_clients = client_coords[client_assignments == facility_idx]
        if len(assigned_clients) > 0:
            plt.scatter(assigned_clients[:, 0], assigned_clients[:, 1], 
                       c=[colors[idx]], marker='o', s=50, alpha=0.6)
            
            # Draw assignment lines
            for client in assigned_clients:
                plt.plot([client[0], facility_coords[facility_idx, 0]], 
                        [client[1], facility_coords[facility_idx, 1]], 
                        c=colors[idx], alpha=0.2, linewidth=0.5)
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("X Coordinate", fontsize=12)
    plt.ylabel("Y Coordinate", fontsize=12)
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.grid(alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.show()

def compare_solutions(instance: dict, 
                     gurobi_solution: Tuple[np.ndarray, np.ndarray, float],
                     vns_solution: Tuple[Tuple[np.ndarray, np.ndarray], List[float]]):
    """
    Compare Gurobi and VNS solutions side by side.
    
    Parameters:
    -----------
    instance : dict
        Dictionary containing the FLP instance data
    gurobi_solution : Tuple[np.ndarray, np.ndarray, float]
        Gurobi solution (facilities, assignments, cost)
    vns_solution : Tuple[Tuple[np.ndarray, np.ndarray], Tuple[List[float], List[int]]]
        VNS solution ((facilities, assignments), (trajectory, trajectory_iterations))
    """
    gurobi_facilities, gurobi_assignments, gurobi_cost = gurobi_solution
    (vns_facilities, vns_assignments), (vns_trajectory, vns_trajectory_iterations) = vns_solution
    
    vns_cost = compute_total_cost(instance['fixed_costs'], instance['service_costs'], 
                                   vns_facilities, vns_assignments)
    
    gap = ((vns_cost - gurobi_cost) / gurobi_cost) * 100
    
    # Get total iterations and number of improvements
    total_iterations = vns_trajectory_iterations[-1] if vns_trajectory_iterations else 0
    num_improvements = len(vns_trajectory) - 1  # Excluding initial solution
    
    print("="*60)
    print("SOLUTION COMPARISON")
    print("="*60)
    print(f"\nGurobi (Exact):")
    print(f"  Total Cost: ${gurobi_cost:.2f}")
    print(f"  Facilities Opened: {gurobi_facilities.sum()}")
    print(f"  Open Facility Indices: {np.where(gurobi_facilities == 1)[0].tolist()}")
    
    print(f"\nVNS (Heuristic):")
    print(f"  Total Cost: ${vns_cost:.2f}")
    print(f"  Facilities Opened: {vns_facilities.sum()}")
    print(f"  Open Facility Indices: {np.where(vns_facilities == 1)[0].tolist()}")
    print(f"  Total Iterations: {total_iterations}")
    print(f"  Improvements Found: {num_improvements}")
    
    print(f"\nComparison:")
    print(f"  Optimality Gap: {gap:.2f}%")
    print(f"  Cost Difference: ${vns_cost - gurobi_cost:.2f}")
    print(f"  Solution Quality: {'Optimal' if gap == 0 else 'Near-optimal' if gap < 5 else 'Good' if gap < 10 else 'Acceptable'}")
    print("="*60)

print("✅ Visualization and Comparison modules loaded successfully.")

# Example usage (uncomment to test)
# Load an instance and solve with both methods
# file_path = "cflp_instances/001_cflp_small_easy_fac15_cli50.json"
# num_facilities, num_clients, fixed_costs, service_costs, facility_capacities, client_demands = load_cflp_instance(file_path)
# instance = {
#     "num_facilities": num_facilities,
#     "num_clients": num_clients,
#     "fixed_costs": fixed_costs,
#     "service_costs": service_costs,
#     "facility_capacities": facility_capacities,
#     "client_demands": client_demands
# }

# Solve with both methods
# gurobi_solution = gurobi_solve_cflp(instance, time_limit=60)
# vns_solution = vns_solve_cflp(instance, max_iterations=50, max_no_improve=10)

# Compare solutions
# compare_solutions(instance, gurobi_solution, vns_solution)

---

## 7. Experiments and Comparison

In this section, we conduct a series of experiments to evaluate and compare the performance of the exact solution using Gurobi and the Variable Neighborhood Search (VNS) heuristic for solving the Capacitated Facility Location Problem (CFLP). The experiments are designed to assess various aspects of both approaches, including solution quality, computational time, convergence behavior, robustness, and scalability.



In [ ]:
def plot_convergence(trajectory: List[float], trajectory_iterations: List[int], 
                    optimal_cost: float, instance_name: str, 
                    output_folder: str, size: str, difficulty: str):
    """
    Plot and save the convergence trajectory of VNS algorithm for a single instance.
    
    Parameters:
    -----------
    trajectory : List[float]
        List of best costs found at improvement iterations
    trajectory_iterations : List[int]
        List of actual iteration numbers where improvements occurred
    optimal_cost : float
        Optimal cost from Gurobi (for reference line)
    instance_name : str
        Name of the instance file
    output_folder : str
        Folder to save the convergence plot
    size : str
        Size category (small, medium, large)
    difficulty : str
        Difficulty level (easy, moderate, expensive)
    """
    # Create figure with professional styling
    plt.figure(figsize=(14, 8))
    plt.style.use('seaborn-v0_8-darkgrid')
    
    # Use actual iteration numbers (not just 0, 1, 2, ...)
    iterations = np.array(trajectory_iterations)
    costs = np.array(trajectory)
    
    # Create step-wise curve that maintains best cost at each iteration (no oscillations!)
    # This creates a "staircase" effect showing cost held constant until next improvement
    iterations_extended = []
    costs_extended = []
    
    for i in range(len(iterations)):
        iterations_extended.append(iterations[i])
        costs_extended.append(costs[i])
        
        # Add horizontal line to next improvement (except for last point)
        if i < len(iterations) - 1:
            iterations_extended.append(iterations[i+1])
            costs_extended.append(costs[i])
    
    # Plot smooth step-wise curve (no artificial waves!)
    plt.plot(iterations_extended, costs_extended, color='#2E86AB', linewidth=3, 
             alpha=0.9, label='VNS Best Cost', zorder=2)
    
    # Overlay actual improvement points with prominent markers
    plt.scatter(iterations, costs, s=120, color='#A23B72', marker='o', 
               edgecolors='#F18F01', linewidths=2.5, alpha=0.95, 
               label='Improvement Points', zorder=3)
    
    # Plot optimal cost reference line with elegant styling
    plt.axhline(y=optimal_cost, color='#C73E1D', linestyle='--', linewidth=2.5, 
                alpha=0.85, label=f'Gurobi Optimal: {optimal_cost:.2f}', zorder=1)
    
    # Fill area between VNS trajectory and optimal cost (step-wise fill)
    plt.fill_between(iterations_extended, costs_extended, optimal_cost, 
                    where=(np.array(costs_extended) >= optimal_cost), 
                    alpha=0.15, color='#A23B72', step='post')
    
    # Calculate and show final gap
    final_gap = ((trajectory[-1] - optimal_cost) / optimal_cost) * 100
    
    # Professional title and labels
    plt.title(f'VNS Convergence Analysis: {instance_name}\n'
              f'Size: {size.capitalize()} | Difficulty: {difficulty.capitalize()} | '
              f'Final Gap: {final_gap:.2f}%',
              fontsize=14, fontweight='bold', pad=20, color='#1A1A1A')
    plt.xlabel('Iteration Number', fontsize=13, fontweight='bold', color='#1A1A1A')
    plt.ylabel('Total Cost', fontsize=13, fontweight='bold', color='#1A1A1A')
    plt.legend(loc='upper right', fontsize=12, framealpha=0.95, shadow=True, 
              fancybox=True, borderpad=1)
    plt.grid(True, alpha=0.3, linestyle='-', linewidth=0.6, color='gray')
    
    # Add subtle vertical lines at improvement iterations
    for iter_num in iterations[1:]:  # Skip initial iteration
        plt.axvline(x=iter_num, color='#06A77D', linestyle=':', linewidth=1, alpha=0.25)
    
    # Set x-axis to show all relevant iterations
    if len(iterations) > 1:
        plt.xlim(-1, max(iterations) + 1)
    
    # Add professional statistics box
    total_improvement = trajectory[0] - trajectory[-1]
    pct_improvement = (total_improvement / trajectory[0]) * 100
    total_iterations = int(iterations[-1])  # Last iteration number
    num_improvements = len(trajectory) - 1  # Number of improvements after initial
    
    # Create elegant info box with metrics
    textstr = f'📊 Performance Metrics\n'
    textstr += f'{"─" * 28}\n'
    textstr += f'Initial Cost:      {trajectory[0]:>12,.2f}\n'
    textstr += f'Final Cost:        {trajectory[-1]:>12,.2f}\n'
    textstr += f'Improvement:       {pct_improvement:>11.2f}%\n'
    textstr += f'{"─" * 28}\n'
    textstr += f'Total Iterations:  {total_iterations:>12,}\n'
    textstr += f'Improvements:      {num_improvements:>12,}'
    
    props = dict(boxstyle='round,pad=0.8', facecolor='#F0F0F0', 
                edgecolor='#2E86AB', linewidth=2, alpha=0.95)
    plt.text(0.02, 0.98, textstr, transform=plt.gca().transAxes, fontsize=10.5,
             verticalalignment='top', bbox=props, family='monospace',
             color='#1A1A1A', weight='medium')
    
    # Add subtle background enhancement
    ax = plt.gca()
    ax.set_facecolor('#FAFAFA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#666666')
    ax.spines['bottom'].set_color('#666666')
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    
    plt.tight_layout()
    
    # Save plot with high resolution for publication quality
    plot_filename = instance_name.replace('.json', '_convergence.png')
    plot_path = os.path.join(output_folder, plot_filename)
    plt.savefig(plot_path, dpi=300, bbox_inches='tight', facecolor='white', 
               edgecolor='none', pad_inches=0.1)
    plt.close()

# Test convergence plot (uncomment to run)
# trajectory_example = [3578.14, 3200.50, 2900.25, 2700.10, 2530.12, 2530.12]
# plot_convergence(trajectory_example, 2500.00, "001_flp_small_easy_fac4_cli49.json", 
#                  "convergence_plots", "small", "easy")


def run_experiments(instances_folder: str = "cflp_instances", 
                   results_file: str = "results_comparison.csv",
                   vns_max_iterations: int = 100,
                   vns_max_no_improve: int = 20,
                   vns_max_neighbors: int = 10,
                   gurobi_time_limit: int = 300,
                   save_convergence_plots: bool = True,
                   convergence_plots_folder: str = "convergence_plots",
                   gurobi_version: str = "free"):
    """
    Run experiments comparing Gurobi and VNS on all instances in the folder.
    
    Parameters:
    -----------
    instances_folder : str
        Folder containing the FLP instances
    results_file : str
        CSV file to save the comparison results
    vns_max_iterations : int
        Maximum number of VNS iterations
    vns_max_no_improve : int
        Maximum number of iterations without improvement before stopping VNS
    vns_max_neighbors : int
        Maximum number of neighbors to generate in each neighborhood structure
    gurobi_time_limit : int
        Time limit for Gurobi solver in seconds
    save_convergence_plots : bool
        Whether to save grouped convergence plots (16 instances per 4x4 grid)
        NOTE: Individual plots are NOT generated to save time and space
    convergence_plots_folder : str
        Folder to save grouped convergence plots (9 PNG files total)
    gurobi_version : str
        Gurobi license type: "academic" (unlimited) or "free" (max 2000 vars/constraints)
        Default: "academic"
        
    Returns:
    --------
    results_df : pd.DataFrame
        DataFrame containing the comparison results for all instances
    """
    # Create convergence plots folder if needed
    if save_convergence_plots:
        os.makedirs(convergence_plots_folder, exist_ok=True)
    
    # Determine if we need to check size limits
    check_size_limit = (gurobi_version.lower() == "free")
    
    # Get all instance files
    instance_files = [f for f in os.listdir(instances_folder) if f.endswith('.json')]
    instance_files.sort()
    
    results = []
    convergence_data = {}  # Store convergence data for grouped plots
    skipped_instances = 0
    
    print(f"Running experiments on {len(instance_files)} instances...")
    print(f"Gurobi version: {gurobi_version.upper()}")
    if check_size_limit:
        print("⚠️  Free version limits: max 2000 variables and 2000 constraints")
        print("    Large instances exceeding these limits will be skipped for Gurobi.\n")
    else:
        print("✓ Academic version: unlimited variables and constraints\n")
    
    for idx, instance_file in enumerate(instance_files, 1):
        instance_path = os.path.join(instances_folder, instance_file)
        print(f"[{idx}/{len(instance_files)}] Processing {instance_file}")
        
        # Load instance
        num_facilities, num_clients, fixed_costs, service_costs, facility_capacities, client_demands = load_cflp_instance(instance_path)
        instance = {
            "num_facilities": num_facilities,
            "num_clients": num_clients,
            "fixed_costs": fixed_costs,
            "service_costs": service_costs,
            "facility_capacities": facility_capacities,
            "client_demands": client_demands
        }
        
        # Extract metadata from filename
        parts = instance_file.replace('.json', '').split('_')
        instance_id = int(parts[0])
        size = parts[2]
        difficulty = parts[3]
        
        # Run Gurobi
        print("  - Running Gurobi...")
        try:
            gurobi_start = time.time()
            gurobi_facilities, gurobi_assignments, gurobi_cost = gurobi_solve_cflp(
                instance, time_limit=gurobi_time_limit, check_size_limit=check_size_limit
            )
            gurobi_time = time.time() - gurobi_start
            gurobi_status = "Optimal"
        except ValueError as e:
            # Instance exceeds free version limits
            print(f"    ⚠️  Skipped (exceeds free version limits)")
            gurobi_cost = np.nan
            gurobi_time = np.nan
            gurobi_status = "Skipped (Size Limit)"
            skipped_instances += 1
        except Exception as e:
            print(f"    Gurobi failed: {str(e)}")
            gurobi_cost = np.inf
            gurobi_time = gurobi_time_limit
            gurobi_status = "Failed"
        
        # Run VNS
        print("  - Running VNS...")
        try:
            vns_start = time.time()
            vns_solution, (vns_trajectory, vns_trajectory_iterations) = vns_solve_cflp(
                instance,
                max_iterations=vns_max_iterations,
                max_no_improve=vns_max_no_improve,
                max_neighbors=vns_max_neighbors,
                neighborhood_structures=['n1', 'n2', 'n3', 'n4']
            )
            vns_time = time.time() - vns_start
            vns_facilities, vns_assignments = vns_solution
            vns_cost = compute_total_cost(fixed_costs, service_costs, vns_facilities, vns_assignments)
            vns_status = "Completed"
            vns_iterations = vns_trajectory_iterations[-1] if vns_trajectory_iterations else 0  # Total iterations
            
            # Store convergence data for grouped plotting (skip individual plots)
            if save_convergence_plots and gurobi_status == "Optimal":
                convergence_data[instance_file] = (vns_trajectory, vns_trajectory_iterations, gurobi_cost)
        except Exception as e:
            print(f"    VNS failed: {str(e)}")
            vns_cost = np.inf
            vns_time = 0
            vns_status = "Failed"
            vns_iterations = 0
            vns_trajectory = []
            vns_trajectory_iterations = []
        
        # Calculate gap (only if Gurobi solved the instance)
        if gurobi_status == "Optimal" and gurobi_cost > 0 and gurobi_cost != np.inf:
            gap_percent = ((vns_cost - gurobi_cost) / gurobi_cost) * 100
        else:
            gap_percent = np.nan
        
        # Store results
        results.append({
            'instance_id': instance_id,
            'instance_name': instance_file,
            'size': size,
            'difficulty': difficulty,
            'num_facilities': num_facilities,
            'num_clients': num_clients,
            'gurobi_cost': gurobi_cost,
            'gurobi_time': gurobi_time,
            'gurobi_status': gurobi_status,
            'vns_cost': vns_cost,
            'vns_time': vns_time,
            'vns_iterations': vns_iterations,
            'vns_status': vns_status,
            'gap_percent': gap_percent,
            'speedup_factor': gurobi_time / vns_time if vns_time > 0 else np.nan
        })
        
        if gurobi_status == "Skipped (Size Limit)":
            print(f"  - Gurobi: {gurobi_status}")
            print(f"  - VNS: Cost={vns_cost:.2f}, Time={vns_time:.2f}s\n")
        else:
            print(f"  - Gurobi: Cost={gurobi_cost:.2f}, Time={gurobi_time:.2f}s")
            print(f"  - VNS: Cost={vns_cost:.2f}, Time={vns_time:.2f}s, Gap={gap_percent:.2f}%\n")
    
    # Create DataFrame
    results_df = pd.DataFrame(results)
    
    # Print summary
    print("\n" + "="*80)
    print("EXPERIMENTS SUMMARY")
    print("="*80)
    print(f"Total instances: {len(instance_files)}")
    print(f"Gurobi version: {gurobi_version.upper()}")
    if check_size_limit:
        solved_by_gurobi = len([r for r in results if r['gurobi_status'] == 'Optimal'])
        print(f"✓ Solved by Gurobi: {solved_by_gurobi}")
        print(f"⚠️  Skipped by Gurobi (size limit): {skipped_instances}")
        print(f"   (Analysis will be performed only on solved instances)")
    else:
        print(f"✓ All instances solved by Gurobi")
    print(f"✓ All instances solved by VNS: {len([r for r in results if r['vns_status'] == 'Completed'])}")
    print("="*80 + "\n")
    
    # Save to CSV
    results_df.to_csv(results_file, index=False)
    print(f"\nResults saved to {results_file}")
    
    # Also save to XLSX with multiple sheets for better analysis
    xlsx_file = results_file.replace('.csv', '.xlsx')
    with pd.ExcelWriter(xlsx_file, engine='openpyxl') as writer:
        # Main results
        results_df.to_excel(writer, sheet_name='All Results', index=False)
        
        # Summary by size
        size_summary = results_df.groupby('size').agg({
            'gurobi_cost': ['mean', 'std'],
            'vns_cost': ['mean', 'std'],
            'gap_percent': ['mean', 'min', 'max'],
            'gurobi_time': ['mean', 'std'],
            'vns_time': ['mean', 'std'],
            'speedup_factor': ['mean', 'std'],
            'instance_id': 'count'
        }).round(2)
        size_summary.to_excel(writer, sheet_name='Summary by Size')
        
        # Summary by difficulty
        diff_summary = results_df.groupby('difficulty').agg({
            'gurobi_cost': ['mean', 'std'],
            'vns_cost': ['mean', 'std'],
            'gap_percent': ['mean', 'min', 'max'],
            'gurobi_time': ['mean', 'std'],
            'vns_time': ['mean', 'std'],
            'speedup_factor': ['mean', 'std'],
            'instance_id': 'count'
        }).round(2)
        diff_summary.to_excel(writer, sheet_name='Summary by Difficulty')
        
        # Size-Difficulty matrix
        pivot_gap = results_df.pivot_table(
            values='gap_percent', 
            index='size', 
            columns='difficulty', 
            aggfunc='mean'
        ).round(2)
        pivot_gap.to_excel(writer, sheet_name='Gap Matrix')
        
        pivot_speedup = results_df.pivot_table(
            values='speedup_factor', 
            index='size', 
            columns='difficulty', 
            aggfunc='mean'
        ).round(2)
        pivot_speedup.to_excel(writer, sheet_name='Speedup Matrix')
    
    print(f"Results also saved to {xlsx_file} with multiple analysis sheets")
    
    # Generate grouped convergence plots (16 instances per grid - ONLY these, no individual plots)
    if save_convergence_plots and convergence_data:
        print("\n" + "="*80)
        print("GENERATING GROUPED CONVERGENCE PLOTS (4x4 grids - 16 per image)")
        print("="*80)
        plot_grouped_convergence(results_df, convergence_data, convergence_plots_folder)
    
    return results_df

print("✅ Plot convergence and run experiments modules loaded successfully.")


def plot_grouped_convergence(results_df: pd.DataFrame, 
                             convergence_data: dict,
                             output_folder: str = "convergence_plots_grouped"):
    """
    Create grouped convergence plots showing 16 instances per image,
    organized by size and difficulty in a 4x4 grid.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame containing experiment results with columns: instance_name, size, difficulty
    convergence_data : dict
        Dictionary mapping instance_name to (trajectory, trajectory_iterations, optimal_cost)
    output_folder : str
        Folder to save the grouped convergence plots
    """
    os.makedirs(output_folder, exist_ok=True)
    
    # Group instances by size and difficulty
    size_order = ['small', 'medium', 'large']
    difficulty_order = ['easy', 'moderate', 'expensive']
    
    for size in size_order:
        for difficulty in difficulty_order:
            # Filter instances for this size/difficulty combination
            mask = (results_df['size'] == size) & (results_df['difficulty'] == difficulty)
            instances = results_df[mask]['instance_name'].tolist()
            
            if not instances:
                continue
            
            # Limit to 16 instances per plot
            instances = instances[:16]
            num_instances = len(instances)
            
            # Create 4x4 subplot grid
            fig, axes = plt.subplots(4, 4, figsize=(20, 20))
            fig.suptitle(f'VNS Convergence: {size.upper()} instances - {difficulty.upper()} difficulty',
                        fontsize=18, fontweight='bold', y=0.995)
            
            axes = axes.flatten()
            
            for idx, instance_name in enumerate(instances):
                ax = axes[idx]
                
                # Get convergence data for this instance
                if instance_name not in convergence_data:
                    ax.axis('off')
                    continue
                
                trajectory, trajectory_iterations, optimal_cost = convergence_data[instance_name]
                
                if not trajectory or not trajectory_iterations:
                    ax.axis('off')
                    continue
                
                # Handle case where VNS found no improvements (only initial solution)
                # Add a final data point at max_iterations to show flat line
                if len(trajectory) == 1:
                    trajectory = trajectory + [trajectory[0]]  # Duplicate final cost
                    trajectory_iterations = trajectory_iterations + [100]  # Assume max_iterations=100
                
                iterations = np.array(trajectory_iterations)
                costs = np.array(trajectory)
                
                # Create step-wise curve for best solution line
                iterations_extended = []
                costs_extended = []
                
                for i in range(len(iterations)):
                    iterations_extended.append(iterations[i])
                    costs_extended.append(costs[i])
                    
                    if i < len(iterations) - 1:
                        iterations_extended.append(iterations[i+1])
                        costs_extended.append(costs[i])
                
                # STYLE LIKE THE REFERENCE IMAGE:
                
                # 1. Plot FADED background trajectory (lighter, transparent)
                ax.plot(iterations_extended, costs_extended, color='#FF6B6B', 
                       linewidth=1.5, alpha=0.25, zorder=1, linestyle='-')
                
                # 2. Plot MAIN SOLID curve (darker, prominent)
                ax.plot(iterations_extended, costs_extended, color='#D32F2F', 
                       linewidth=2.5, alpha=0.9, zorder=2, label='VNS Best Solution')
                
                # 3. Plot optimal/best solution reference line
                ax.axhline(y=optimal_cost, color='#1976D2', linestyle='--', 
                          linewidth=1.5, alpha=0.8, zorder=3, label='Gurobi Optimal')
                
                # Calculate final gap
                final_gap = ((costs[-1] - optimal_cost) / optimal_cost) * 100
                
                # Set title with instance info
                instance_short = instance_name.replace('.json', '').split('_')[-2:]
                title = f"{instance_short[0]}{instance_short[1]}\nFinal: {final_gap:.1f}%"
                ax.set_title(title, fontsize=9, fontweight='bold', pad=8)
                
                # Add legend in upper-left corner (like reference image)
                ax.legend(loc='upper left', fontsize=6, framealpha=0.9, 
                         edgecolor='gray', fancybox=False)
                
                # Style - clean grid
                ax.grid(True, alpha=0.25, linestyle='-', linewidth=0.5, color='gray')
                ax.set_xlabel('Iteration', fontsize=8)
                ax.set_ylabel('Cost', fontsize=8)
                ax.tick_params(labelsize=7)
                
                # Clean background
                ax.set_facecolor('white')
                
                # Set x-axis limits
                if len(iterations) > 1:
                    ax.set_xlim(-1, max(iterations) * 1.05)
            
            # Hide unused subplots
            for idx in range(num_instances, 16):
                axes[idx].axis('off')
            
            plt.tight_layout(rect=[0, 0, 1, 0.99])
            
            # Save plot
            output_file = os.path.join(output_folder, 
                                      f'convergence_grid_{size}_{difficulty}.png')
            plt.savefig(output_file, dpi=300, bbox_inches='tight', 
                       facecolor='white', edgecolor='none')
            plt.close()
            
            print(f"✓ Saved grouped plot: {size}_{difficulty} ({num_instances} instances)")
    
    print(f"\n✅ All grouped convergence plots saved to '{output_folder}/'")

print("✅ Grouped convergence plotting module loaded successfully.")

# Run experiments (uncomment to execute)
# results_df = run_experiments(
#     instances_folder="cflp_instances",
#     results_file="results_comparison.csv",
#     vns_max_iterations=100,
#     vns_max_no_improve=20,
#     vns_max_neighbors=10,
#     gurobi_time_limit=300
# )

---

## 8. Visualization and Analysis

In this section, we visualize and analyze the results obtained from both the exact solution using Gurobi and the Variable Neighborhood Search (VNS) heuristic for the Facility Location Problem (FLP). Visualization helps in understanding the performance differences between the two approaches, while analysis provides insights into their strengths and weaknesses.

In [ ]:
def visualize_results(results_df: pd.DataFrame, output_file: str = "comparison_analysis.png"):
    """
    Create comprehensive visualizations comparing Gurobi and VNS performance.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame containing the comparison results
    output_file : str
        Output filename for the visualization (default: "comparison_analysis.png")
    """
    # Set up the plotting style
    plt.style.use('seaborn-v0_8-whitegrid')
    sns.set_palette("husl")
    
    # Filter valid results (no failures)
    valid_results = results_df[
        (results_df['gurobi_status'] == 'Optimal') & 
        (results_df['vns_status'] == 'Completed')
    ].copy()
    
    # Create a figure with multiple subplots
    fig = plt.figure(figsize=(20, 12))
    
    # 1. Solution Quality Comparison (Gap Distribution)
    ax1 = plt.subplot(2, 3, 1)
    sns.boxplot(data=valid_results, x='size', y='gap_percent', hue='difficulty', ax=ax1)
    ax1.set_title('Solution Quality: Optimality Gap Distribution', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Instance Size', fontsize=12)
    ax1.set_ylabel('Gap from Optimal (%)', fontsize=12)
    ax1.legend(title='Difficulty', loc='upper left')
    ax1.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
    ax1.grid(True, alpha=0.3)
    
    # 2. Computational Time Comparison
    ax2 = plt.subplot(2, 3, 2)
    time_comparison = valid_results.melt(
        id_vars=['size', 'difficulty'],
        value_vars=['gurobi_time', 'vns_time'],
        var_name='Method',
        value_name='Time (s)'
    )
    time_comparison['Method'] = time_comparison['Method'].map({'gurobi_time': 'Gurobi', 'vns_time': 'VNS'})
    sns.boxplot(data=time_comparison, x='size', y='Time (s)', hue='Method', ax=ax2)
    ax2.set_title('Computational Time Comparison', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Instance Size', fontsize=12)
    ax2.set_ylabel('Time (seconds)', fontsize=12)
    ax2.set_yscale('log')
    ax2.legend(title='Method')
    ax2.grid(True, alpha=0.3)
    
    # 3. Speedup Factor
    ax3 = plt.subplot(2, 3, 3)
    sns.boxplot(data=valid_results, x='size', y='speedup_factor', hue='difficulty', ax=ax3)
    ax3.set_title('VNS Speedup Factor (Gurobi Time / VNS Time)', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Instance Size', fontsize=12)
    ax3.set_ylabel('Speedup Factor', fontsize=12)
    ax3.axhline(y=1, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Equal Performance')
    ax3.legend(title='Difficulty', loc='upper left')
    ax3.grid(True, alpha=0.3)
    
    # 4. Scalability: Time vs Problem Size
    ax4 = plt.subplot(2, 3, 4)
    valid_results['problem_size'] = valid_results['num_facilities'] * valid_results['num_clients']
    ax4.scatter(valid_results['problem_size'], valid_results['gurobi_time'], 
                alpha=0.6, s=100, label='Gurobi', marker='o')
    ax4.scatter(valid_results['problem_size'], valid_results['vns_time'], 
                alpha=0.6, s=100, label='VNS', marker='^')
    ax4.set_title('Scalability: Time vs Problem Size', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Problem Size (Facilities × Clients)', fontsize=12)
    ax4.set_ylabel('Time (seconds)', fontsize=12)
    ax4.set_xscale('log')
    ax4.set_yscale('log')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Gap vs Problem Size
    ax5 = plt.subplot(2, 3, 5)
    scatter = ax5.scatter(valid_results['problem_size'], valid_results['gap_percent'],
                          c=valid_results['difficulty'].map({'easy': 0, 'moderate': 1, 'expensive': 2}),
                          cmap='viridis', alpha=0.6, s=100)
    ax5.set_title('Solution Quality vs Problem Size', fontsize=14, fontweight='bold')
    ax5.set_xlabel('Problem Size (Facilities × Clients)', fontsize=12)
    ax5.set_ylabel('Gap from Optimal (%)', fontsize=12)
    ax5.set_xscale('log')
    ax5.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.7)
    cbar = plt.colorbar(scatter, ax=ax5, ticks=[0, 1, 2])
    cbar.set_label('Difficulty', fontsize=10)
    cbar.ax.set_yticklabels(['Easy', 'Moderate', 'Expensive'])
    ax5.grid(True, alpha=0.3)
    
    # 6. Summary Statistics Table
    ax6 = plt.subplot(2, 3, 6)
    ax6.axis('tight')
    ax6.axis('off')
    
    summary_stats = pd.DataFrame({
        'Metric': [
            'Total Instances',
            'Avg Gap (%)',
            'Max Gap (%)',
            'Min Gap (%)',
            'Avg Gurobi Time (s)',
            'Avg VNS Time (s)',
            'Avg Speedup Factor',
            'VNS Faster (%)' 
        ],
        'Value': [
            len(valid_results),
            f"{valid_results['gap_percent'].mean():.2f}",
            f"{valid_results['gap_percent'].max():.2f}",
            f"{valid_results['gap_percent'].min():.2f}",
            f"{valid_results['gurobi_time'].mean():.2f}",
            f"{valid_results['vns_time'].mean():.2f}",
            f"{valid_results['speedup_factor'].mean():.2f}",
            f"{(valid_results['speedup_factor'] > 1).sum() / len(valid_results) * 100:.1f}"
        ]
    })
    
    table = ax6.table(cellText=summary_stats.values, colLabels=summary_stats.columns,
                      cellLoc='left', loc='center', colWidths=[0.6, 0.4])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    ax6.set_title('Summary Statistics', fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"Visualization saved as '{output_file}'")
    plt.show()
    
    # Additional detailed analysis by size and difficulty
    print("\n" + "="*80)
    print("DETAILED ANALYSIS BY SIZE AND DIFFICULTY")
    print("="*80)
    
    for size in ['small', 'medium', 'large']:
        size_data = valid_results[valid_results['size'] == size]
        if len(size_data) > 0:
            print(f"\n{size.upper()} Instances:")
            print(f"  Count: {len(size_data)}")
            print(f"  Avg Gap: {size_data['gap_percent'].mean():.2f}%")
            print(f"  Avg Gurobi Time: {size_data['gurobi_time'].mean():.2f}s")
            print(f"  Avg VNS Time: {size_data['vns_time'].mean():.2f}s")
            print(f"  Avg Speedup: {size_data['speedup_factor'].mean():.2f}x")
            
            for difficulty in ['easy', 'moderate', 'expensive']:
                subset = size_data[size_data['difficulty'] == difficulty]
                if len(subset) > 0:
                    print(f"    {difficulty.capitalize()}: Gap={subset['gap_percent'].mean():.2f}%, "
                          f"Speedup={subset['speedup_factor'].mean():.2f}x")

print("✅ Results visualization module loaded successfully.")

# Example usage (uncomment to run)
# results_df = pd.read_csv('results_comparison.csv')
# visualize_results(results_df)

def create_comprehensive_analysis_plots(results_df: pd.DataFrame, output_folder: str = "analysis_plots"):
    """
    Create comprehensive analysis plots for student report writing.
    
    These plots enable students to analyze and discuss:
    - Performance across different problem sizes
    - Impact of difficulty levels
    - Trade-offs between solution quality and computation time
    - Scalability patterns
    - Algorithm behavior under various conditions
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame containing the comparison results
    output_folder : str
        Folder to save individual analysis plots
    """
    # Create output folder
    os.makedirs(output_folder, exist_ok=True)
    
    # Filter valid results
    valid_results = results_df[
        (results_df['gurobi_status'] == 'Optimal') & 
        (results_df['vns_status'] == 'Completed')
    ].copy()
    
    valid_results['problem_size'] = valid_results['num_facilities'] * valid_results['num_clients']
    
    # Set modern attractive color palette
    plt.style.use('seaborn-v0_8-whitegrid')
    
    # Define attractive color palettes
    size_colors = {'small': '#4ECDC4', 'medium': '#FF6B6B', 'large': '#95E1D3'}
    difficulty_colors = {'easy': '#38B6FF', 'moderate': '#FF8C42', 'expensive': '#E63946'}
    method_colors = {'Gurobi': '#5E60CE', 'VNS': '#FF6D00'}
    
    print("Generating comprehensive analysis plots for student reports...")
    print("="*80)
    
    # ============================================================================
    # PLOT 1: Gap Analysis by Size and Difficulty (Detailed Boxplot)
    # ============================================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.patch.set_facecolor('white')
    
    # By Size - with attractive colors
    sns.boxplot(data=valid_results, x='size', y='gap_percent', hue='difficulty', 
               palette=difficulty_colors, ax=axes[0])
    axes[0].set_title('Optimality Gap by Problem Size and Difficulty', 
                     fontsize=14, fontweight='bold', color='#2C3E50')
    axes[0].set_xlabel('Problem Size', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Gap from Optimal (%)', fontsize=12, fontweight='bold')
    axes[0].legend(title='Difficulty', loc='upper left', framealpha=0.95, 
                  edgecolor='#2C3E50', fancybox=True)
    axes[0].axhline(y=0, color='#E63946', linestyle='--', linewidth=2, alpha=0.7)
    axes[0].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[0].set_facecolor('#F8F9FA')
    
    # By Difficulty - with attractive colors
    sns.boxplot(data=valid_results, x='difficulty', y='gap_percent', hue='size', 
               palette=size_colors, ax=axes[1])
    axes[1].set_title('Optimality Gap by Difficulty and Problem Size', 
                     fontsize=14, fontweight='bold', color='#2C3E50')
    axes[1].set_xlabel('Difficulty Level', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Gap from Optimal (%)', fontsize=12, fontweight='bold')
    axes[1].legend(title='Size', loc='upper left', framealpha=0.95, 
                  edgecolor='#2C3E50', fancybox=True)
    axes[1].axhline(y=0, color='#E63946', linestyle='--', linewidth=2, alpha=0.7)
    axes[1].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[1].set_facecolor('#F8F9FA')
    
    plt.tight_layout()
    plt.savefig(f'{output_folder}/01_gap_analysis_by_categories.png', 
               dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print("✓ Saved: 01_gap_analysis_by_categories.png")
    
    # ============================================================================
    # PLOT 2: Computation Time Comparison (Detailed)
    # ============================================================================
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor('white')
    
    # Time by Size
    time_data_size = valid_results.melt(
        id_vars=['size', 'difficulty'],
        value_vars=['gurobi_time', 'vns_time'],
        var_name='Method',
        value_name='Time (s)'
    )
    time_data_size['Method'] = time_data_size['Method'].map({'gurobi_time': 'Gurobi', 'vns_time': 'VNS'})
    
    sns.boxplot(data=time_data_size, x='size', y='Time (s)', hue='Method', 
               palette=method_colors, ax=axes[0, 0])
    axes[0, 0].set_title('Computation Time by Problem Size', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[0, 0].set_xlabel('Problem Size', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylabel('Time (seconds, log scale)', fontsize=11, fontweight='bold')
    axes[0, 0].set_yscale('log')
    axes[0, 0].legend(title='Method', framealpha=0.95, edgecolor='#2C3E50')
    axes[0, 0].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[0, 0].set_facecolor('#F8F9FA')
    
    # Time by Difficulty
    time_data_diff = valid_results.melt(
        id_vars=['size', 'difficulty'],
        value_vars=['gurobi_time', 'vns_time'],
        var_name='Method',
        value_name='Time (s)'
    )
    time_data_diff['Method'] = time_data_diff['Method'].map({'gurobi_time': 'Gurobi', 'vns_time': 'VNS'})
    
    sns.boxplot(data=time_data_diff, x='difficulty', y='Time (s)', hue='Method', 
               palette=method_colors, ax=axes[0, 1])
    axes[0, 1].set_title('Computation Time by Difficulty Level', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[0, 1].set_xlabel('Difficulty Level', fontsize=11, fontweight='bold')
    axes[0, 1].set_ylabel('Time (seconds, log scale)', fontsize=11, fontweight='bold')
    axes[0, 1].set_yscale('log')
    axes[0, 1].legend(title='Method', framealpha=0.95, edgecolor='#2C3E50')
    axes[0, 1].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[0, 1].set_facecolor('#F8F9FA')
    
    # Gurobi detailed by size and difficulty - attractive gradient
    pivot_gurobi = valid_results.pivot_table(
        values='gurobi_time', 
        index='size', 
        columns='difficulty', 
        aggfunc='mean'
    )
    sns.heatmap(pivot_gurobi, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=axes[1, 0], 
               cbar_kws={'label': 'Avg Time (s)'}, linewidths=1, linecolor='white')
    axes[1, 0].set_title('Gurobi: Average Time (Size × Difficulty)', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[1, 0].set_xlabel('Difficulty Level', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('Problem Size', fontsize=11, fontweight='bold')
    
    # VNS detailed by size and difficulty - attractive gradient
    pivot_vns = valid_results.pivot_table(
        values='vns_time', 
        index='size', 
        columns='difficulty', 
        aggfunc='mean'
    )
    sns.heatmap(pivot_vns, annot=True, fmt='.2f', cmap='YlGnBu', ax=axes[1, 1], 
               cbar_kws={'label': 'Avg Time (s)'}, linewidths=1, linecolor='white')
    axes[1, 1].set_title('VNS: Average Time (Size × Difficulty)', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[1, 1].set_xlabel('Difficulty Level', fontsize=11, fontweight='bold')
    axes[1, 1].set_ylabel('Problem Size', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{output_folder}/02_time_analysis_detailed.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 02_time_analysis_detailed.png")
    
    # ============================================================================
    # PLOT 3: Speedup Analysis (Comprehensive)
    # ============================================================================
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor('white')
    
    # Speedup by Size
    sns.boxplot(data=valid_results, x='size', y='speedup_factor', hue='difficulty', 
               palette=difficulty_colors, ax=axes[0, 0])
    axes[0, 0].set_title('VNS Speedup Factor by Size', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[0, 0].set_xlabel('Problem Size', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylabel('Speedup Factor (Gurobi/VNS)', fontsize=11, fontweight='bold')
    axes[0, 0].axhline(y=1, color='#E63946', linestyle='--', linewidth=2, alpha=0.7, label='Equal')
    axes[0, 0].legend(title='Difficulty', loc='upper left', framealpha=0.95, edgecolor='#2C3E50')
    axes[0, 0].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[0, 0].set_facecolor('#F8F9FA')
    
    # Speedup by Difficulty
    sns.boxplot(data=valid_results, x='difficulty', y='speedup_factor', hue='size', 
               palette=size_colors, ax=axes[0, 1])
    axes[0, 1].set_title('VNS Speedup Factor by Difficulty', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[0, 1].set_xlabel('Difficulty Level', fontsize=11, fontweight='bold')
    axes[0, 1].set_ylabel('Speedup Factor (Gurobi/VNS)', fontsize=11, fontweight='bold')
    axes[0, 1].axhline(y=1, color='#E63946', linestyle='--', linewidth=2, alpha=0.7, label='Equal')
    axes[0, 1].legend(title='Size', loc='upper left', framealpha=0.95, edgecolor='#2C3E50')
    axes[0, 1].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[0, 1].set_facecolor('#F8F9FA')
    
    # Speedup heatmap - attractive gradient
    pivot_speedup = valid_results.pivot_table(
        values='speedup_factor', 
        index='size', 
        columns='difficulty', 
        aggfunc='mean'
    )
    sns.heatmap(pivot_speedup, annot=True, fmt='.2f', cmap='RdYlGn', center=1.0, 
                ax=axes[1, 0], cbar_kws={'label': 'Avg Speedup'}, vmin=0,
                linewidths=1, linecolor='gray')
    axes[1, 0].set_title('Average Speedup Factor (Size × Difficulty)', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[1, 0].set_xlabel('Difficulty Level', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('Problem Size', fontsize=11, fontweight='bold')
    
    # Speedup vs Problem Size scatter - attractive colors
    difficulty_scatter_colors = {'easy': '#38B6FF', 'moderate': '#FF8C42', 'expensive': '#E63946'}
    for difficulty in valid_results['difficulty'].unique():
        subset = valid_results[valid_results['difficulty'] == difficulty]
        axes[1, 1].scatter(subset['problem_size'], subset['speedup_factor'], 
                          label=difficulty.capitalize(), alpha=0.7, s=80,
                          color=difficulty_scatter_colors[difficulty], edgecolors='gray', linewidth=0.5)
    axes[1, 1].set_title('Speedup vs Problem Size', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[1, 1].set_xlabel('Problem Size (Facilities × Clients)', fontsize=11, fontweight='bold')
    axes[1, 1].set_ylabel('Speedup Factor', fontsize=11, fontweight='bold')
    axes[1, 1].set_xscale('log')
    axes[1, 1].axhline(y=1, color='#E63946', linestyle='--', linewidth=2, alpha=0.7)
    axes[1, 1].legend(title='Difficulty', framealpha=0.95, edgecolor='#2C3E50')
    axes[1, 1].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[1, 1].set_facecolor('#F8F9FA')
    
    plt.tight_layout()
    plt.savefig(f'{output_folder}/03_speedup_analysis_comprehensive.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 03_speedup_analysis_comprehensive.png")
    
    # ============================================================================
    # PLOT 4: Scalability Analysis
    # ============================================================================
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor('white')
    
    # Scalability: Time vs Problem Size (by method) - attractive colors
    for method, color, marker, label in [('gurobi_time', '#5E60CE', 'o', 'Gurobi'), 
                                         ('vns_time', '#FF6D00', '^', 'VNS')]:
        axes[0, 0].scatter(valid_results['problem_size'], valid_results[method],
                          alpha=0.7, s=100, c=color, marker=marker, label=label,
                          edgecolors='gray', linewidth=0.5)
    axes[0, 0].set_title('Scalability: Computation Time vs Problem Size', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[0, 0].set_xlabel('Problem Size (log scale)', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylabel('Time in seconds (log scale)', fontsize=11, fontweight='bold')
    axes[0, 0].set_xscale('log')
    axes[0, 0].set_yscale('log')
    axes[0, 0].legend(framealpha=0.95, edgecolor='#2C3E50')
    axes[0, 0].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[0, 0].set_facecolor('#F8F9FA')
    
    # Gap vs Problem Size - attractive difficulty colors
    difficulty_scatter_colors = {'easy': '#38B6FF', 'moderate': '#FF8C42', 'expensive': '#E63946'}
    for difficulty in valid_results['difficulty'].unique():
        subset = valid_results[valid_results['difficulty'] == difficulty]
        axes[0, 1].scatter(subset['problem_size'], subset['gap_percent'],
                          label=difficulty.capitalize(), alpha=0.7, s=80,
                          color=difficulty_scatter_colors[difficulty], 
                          edgecolors='gray', linewidth=0.5)
    axes[0, 1].set_title('Solution Quality vs Problem Size', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[0, 1].set_xlabel('Problem Size (log scale)', fontsize=11, fontweight='bold')
    axes[0, 1].set_ylabel('Gap from Optimal (%)', fontsize=11, fontweight='bold')
    axes[0, 1].set_xscale('log')
    axes[0, 1].axhline(y=0, color='#E63946', linestyle='--', linewidth=2, alpha=0.7)
    axes[0, 1].legend(title='Difficulty', framealpha=0.95, edgecolor='#2C3E50')
    axes[0, 1].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[0, 1].set_facecolor('#F8F9FA')
    
    # VNS Iterations vs Problem Size - attractive gradient
    difficulty_map = {'easy': 0, 'moderate': 1, 'expensive': 2}
    scatter = axes[1, 0].scatter(valid_results['problem_size'], valid_results['vns_iterations'],
                      c=valid_results['difficulty'].map(difficulty_map),
                      cmap='plasma', alpha=0.7, s=80, edgecolors='gray', linewidth=0.5)
    axes[1, 0].set_title('VNS Iterations vs Problem Size', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[1, 0].set_xlabel('Problem Size (log scale)', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('Number of Iterations', fontsize=11, fontweight='bold')
    axes[1, 0].set_xscale('log')
    axes[1, 0].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[1, 0].set_facecolor('#F8F9FA')
    
    # Time ratio trend - attractive size colors
    valid_results['time_ratio'] = valid_results['vns_time'] / valid_results['gurobi_time']
    size_scatter_colors = {'small': '#4ECDC4', 'medium': '#FF6B6B', 'large': '#95E1D3'}
    for size in valid_results['size'].unique():
        subset = valid_results[valid_results['size'] == size]
        axes[1, 1].scatter(subset['problem_size'], subset['time_ratio'],
                          label=size.capitalize(), alpha=0.7, s=80,
                          color=size_scatter_colors[size], 
                          edgecolors='gray', linewidth=0.5)
    axes[1, 1].set_title('Time Efficiency: VNS/Gurobi Ratio', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[1, 1].set_xlabel('Problem Size (log scale)', fontsize=11, fontweight='bold')
    axes[1, 1].set_ylabel('Time Ratio (VNS/Gurobi)', fontsize=11, fontweight='bold')
    axes[1, 1].set_xscale('log')
    axes[1, 1].set_yscale('log')
    axes[1, 1].axhline(y=1, color='#E63946', linestyle='--', linewidth=2, alpha=0.7)
    axes[1, 1].legend(title='Size', framealpha=0.95, edgecolor='#2C3E50')
    axes[1, 1].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[1, 1].set_facecolor('#F8F9FA')
    
    plt.tight_layout()
    plt.savefig(f'{output_folder}/04_scalability_analysis.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 04_scalability_analysis.png")
    
    # ============================================================================
    # PLOT 5: Quality-Time Trade-off Analysis
    # ============================================================================
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor('white')
    
    # Gap vs VNS Time (by size) - attractive colors
    size_scatter_colors = {'small': '#4ECDC4', 'medium': '#FF6B6B', 'large': '#95E1D3'}
    for size in valid_results['size'].unique():
        subset = valid_results[valid_results['size'] == size]
        axes[0, 0].scatter(subset['vns_time'], subset['gap_percent'],
                          label=size.capitalize(), alpha=0.7, s=80,
                          color=size_scatter_colors[size], 
                          edgecolors='gray', linewidth=0.5)
    axes[0, 0].set_title('Quality-Time Trade-off: Gap vs VNS Time', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[0, 0].set_xlabel('VNS Time (seconds)', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylabel('Gap from Optimal (%)', fontsize=11, fontweight='bold')
    axes[0, 0].legend(title='Size', framealpha=0.95, edgecolor='#2C3E50')
    axes[0, 0].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[0, 0].set_facecolor('#F8F9FA')
    
    # Gap vs Speedup (by difficulty) - attractive colors
    difficulty_scatter_colors = {'easy': '#38B6FF', 'moderate': '#FF8C42', 'expensive': '#E63946'}
    for difficulty in valid_results['difficulty'].unique():
        subset = valid_results[valid_results['difficulty'] == difficulty]
        axes[0, 1].scatter(subset['speedup_factor'], subset['gap_percent'],
                          label=difficulty.capitalize(), alpha=0.7, s=80,
                          color=difficulty_scatter_colors[difficulty], 
                          edgecolors='gray', linewidth=0.5)
    axes[0, 1].set_title('Trade-off: Gap vs Speedup Factor', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[0, 1].set_xlabel('Speedup Factor', fontsize=11, fontweight='bold')
    axes[0, 1].set_ylabel('Gap from Optimal (%)', fontsize=11, fontweight='bold')
    axes[0, 1].legend(title='Difficulty', framealpha=0.95, edgecolor='#2C3E50')
    axes[0, 1].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[0, 1].set_facecolor('#F8F9FA')
    
    # Efficiency score: (1 / (1 + gap)) * speedup
    valid_results['efficiency_score'] = (1 / (1 + valid_results['gap_percent']/100)) * valid_results['speedup_factor']
    pivot_efficiency = valid_results.pivot_table(
        values='efficiency_score', 
        index='size', 
        columns='difficulty', 
        aggfunc='mean'
    )
    sns.heatmap(pivot_efficiency, annot=True, fmt='.2f', cmap='YlGn', 
                ax=axes[1, 0], cbar_kws={'label': 'Efficiency Score'},
                linewidths=1, linecolor='gray')
    axes[1, 0].set_title('VNS Efficiency Score (Size × Difficulty)', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[1, 0].set_xlabel('Difficulty Level', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('Problem Size', fontsize=11, fontweight='bold')
    
    # Cost comparison - attractive gradient
    size_map = {'small': 0, 'medium': 1, 'large': 2}
    scatter = axes[1, 1].scatter(valid_results['gurobi_cost'], valid_results['vns_cost'],
                      c=valid_results['size'].map(size_map),
                      cmap='cool', alpha=0.7, s=80, edgecolors='gray', linewidth=0.5)
    # Perfect correlation line
    min_cost = min(valid_results['gurobi_cost'].min(), valid_results['vns_cost'].min())
    max_cost = max(valid_results['gurobi_cost'].max(), valid_results['vns_cost'].max())
    axes[1, 1].plot([min_cost, max_cost], [min_cost, max_cost], 
                   color='#E63946', linestyle='--', linewidth=2.5, alpha=0.8, label='Perfect Match')
    axes[1, 1].set_title('Solution Cost: Gurobi vs VNS', 
                        fontsize=13, fontweight='bold', color='#2C3E50')
    axes[1, 1].set_xlabel('Gurobi Optimal Cost', fontsize=11, fontweight='bold')
    axes[1, 1].set_ylabel('VNS Cost', fontsize=11, fontweight='bold')
    axes[1, 1].legend(framealpha=0.95, edgecolor='#2C3E50')
    axes[1, 1].grid(True, alpha=0.25, linestyle='-', linewidth=0.8)
    axes[1, 1].set_facecolor('#F8F9FA')
    
    plt.tight_layout()
    plt.savefig(f'{output_folder}/05_quality_time_tradeoff.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 05_quality_time_tradeoff.png")
    
    # ============================================================================
    # PLOT 6: Statistical Summary Dashboard
    # ============================================================================
    fig = plt.figure(figsize=(18, 10))
    fig.patch.set_facecolor('white')
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # Summary statistics by category
    summary_data = []
    for size in ['small', 'medium', 'large']:
        for diff in ['easy', 'moderate', 'expensive']:
            subset = valid_results[(valid_results['size'] == size) & (valid_results['difficulty'] == diff)]
            if len(subset) > 0:
                summary_data.append({
                    'Size': size.capitalize(),
                    'Difficulty': diff.capitalize(),
                    'Count': len(subset),
                    'Avg Gap (%)': f"{subset['gap_percent'].mean():.2f}",
                    'Avg Gurobi Time (s)': f"{subset['gurobi_time'].mean():.2f}",
                    'Avg VNS Time (s)': f"{subset['vns_time'].mean():.2f}",
                    'Avg Speedup': f"{subset['speedup_factor'].mean():.2f}",
                })
    
    summary_df = pd.DataFrame(summary_data)
    
    ax_table = fig.add_subplot(gs[0:2, :])
    ax_table.axis('tight')
    ax_table.axis('off')
    table = ax_table.table(cellText=summary_df.values, colLabels=summary_df.columns,
                           cellLoc='center', loc='center', colWidths=[0.11, 0.14, 0.09, 0.13, 0.16, 0.15, 0.13])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.8)
    
    # Style header row with attractive colors
    for (i, j), cell in table.get_celld().items():
        if i == 0:  # Header row
            cell.set_facecolor('#5E60CE')
            cell.set_text_props(weight='bold', color='white', fontsize=11)
            cell.set_edgecolor('#2C3E50')
            cell.set_linewidth(1.5)
        else:
            # Alternate row colors for better readability
            if i % 2 == 0:
                cell.set_facecolor('#F8F9FA')
            else:
                cell.set_facecolor('white')
            cell.set_edgecolor('#D3D3D3')
            cell.set_linewidth(0.8)
    
    ax_table.set_title('Detailed Performance Summary by Category', 
                      fontsize=15, fontweight='bold', pad=20, color='#2C3E50')
    
    # Overall statistics with attractive styling
    ax_stats = fig.add_subplot(gs[2, :])
    ax_stats.axis('off')
    
    # Calculate total runtime
    total_gurobi_time = valid_results['gurobi_time'].sum()
    total_vns_time = valid_results['vns_time'].sum()
    total_runtime = total_gurobi_time + total_vns_time
    
    overall_stats = f"""
    OVERALL STATISTICS:
    Total Instances: {len(valid_results)} | Avg Gap: {valid_results['gap_percent'].mean():.2f}% | Avg Speedup: {valid_results['speedup_factor'].mean():.2f}x
    
    Total Runtime: {total_runtime:.2f}s (Gurobi: {total_gurobi_time:.2f}s + VNS: {total_vns_time:.2f}s)
    
    Best Performance: Gap < 5% in {(valid_results['gap_percent'] < 5).sum()} instances ({(valid_results['gap_percent'] < 5).sum()/len(valid_results)*100:.1f}%)
    Excellent Speedup: >10x faster in {(valid_results['speedup_factor'] > 10).sum()} instances ({(valid_results['speedup_factor'] > 10).sum()/len(valid_results)*100:.1f}%)
    
    VNS is Faster: {(valid_results['speedup_factor'] > 1).sum()} out of {len(valid_results)} instances ({(valid_results['speedup_factor'] > 1).sum()/len(valid_results)*100:.1f}%)
    """
    
    ax_stats.text(0.5, 0.5, overall_stats, ha='center', va='center', fontsize=11.5, 
                 bbox=dict(boxstyle='round,pad=1', facecolor='#E3F2FD', alpha=0.9,
                          edgecolor='#2C3E50', linewidth=2), fontweight='bold', color='#2C3E50')
    
    plt.savefig(f'{output_folder}/06_statistical_summary_dashboard.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Saved: 06_statistical_summary_dashboard.png")
    
    print("="*80)
    print(f"✓ All 6 comprehensive analysis plots saved to '{output_folder}/' folder")
    print("  These plots are designed for student report analysis and discussion.")
    print("="*80)

print("✅ Comprehensive analysis plots module loaded successfully.")

# Example usage (uncomment to run)
# results_df = pd.read_csv('results_comparison.csv')
# create_comprehensive_analysis_plots(results_df)

---

## 9. Main Execution

In this section, we execute the main components of the Facility Location Problem (FLP) solution process. This includes loading instances, running both the exact solution using Gurobi and the Variable Neighborhood Search (VNS) heuristic, and collecting results for comparison and analysis.


In [ ]:
def main(student_id: int = None, gurobi_version: str = "academic"):
    """
    Main execution function that orchestrates the entire FLP solution process.
    
    This function:
    1. Generates FLP instances (if not already generated) using student ID as seed
    2. Runs experiments comparing Gurobi and VNS
    3. Visualizes and analyzes the results
    4. Generates comprehensive analysis plots for report writing
    5. Generates a comprehensive report
    
    Parameters:
    -----------
    student_id : int
        Student ID used as random seed for unique results (REQUIRED)
    gurobi_version : str
        Gurobi license type: "academic" (unlimited) or "free" (max 2000 vars/constraints)
        Default: "academic"
        
        - "academic": All instances will be solved by Gurobi (unlimited)
        - "free": Only instances within 2000 variables and 2000 constraints will be solved
                  Large instances will be skipped for Gurobi but still solved by VNS
    """
    if student_id is None:
        print("="*80)
        print("ERROR: Student ID is required!")
        print("="*80)
        print("\nPlease provide your student ID:")
        print("  main(student_id=YOUR_STUDENT_ID)")
        print("\nExample:")
        print("  main(student_id=12345)")
        print("  main(student_id=12345, gurobi_version='free')  # For free Gurobi license")
        print("\nWhy? Each student needs unique results for their report.")
        print("="*80)
        return
    
    print("="*80)
    print("FACILITY LOCATION PROBLEM - ASSIGNMENT 2")
    print("Comparison of Exact (Gurobi) vs Heuristic (VNS) Approaches")
    print("="*80)
    print(f"\n🎓 Student ID: {student_id} (used as random seed for unique results)")
    print(f"📦 Gurobi Version: {gurobi_version.upper()}")
    if gurobi_version.lower() == "free":
        print("   ⚠️  Free version limits: max 2000 variables and 2000 constraints")
        print("   Large instances will be skipped for Gurobi (but solved by VNS)")
    print("="*80)
    
    # ==================================================================================
    # CLEANUP: Remove ALL previous results and instances for fresh run
    # ==================================================================================
    print("\n" + "="*80)
    print("CLEANING UP ALL PREVIOUS RESULTS (Fresh Run)")
    print("="*80)
    
    # Define all folders and files that need cleanup
    instances_folder = f"cflp_instances_student_{student_id}"
    convergence_folder = f"convergence_plots_student_{student_id}"
    analysis_folder = f"analysis_plots_student_{student_id}"
    results_file = f"results_student_{student_id}.csv"
    results_excel = f"results_student_{student_id}.xlsx"
    main_viz_file = f"comparison_analysis_student_{student_id}.png"
    summary_file = f"report_summary_student_{student_id}.txt"
    
    cleanup_items = [
        (instances_folder, "Instance files folder"),
        (convergence_folder, "Convergence plots folder"),
        (analysis_folder, "Analysis plots folder"),
        (results_file, "Results CSV file"),
        (results_excel, "Results Excel file"),
        (main_viz_file, "Main visualization file"),
        (summary_file, "Report summary file")
    ]
    
    total_removed = 0
    for item_path, item_desc in cleanup_items:
        if os.path.exists(item_path):
            try:
                if os.path.isfile(item_path):
                    os.unlink(item_path)
                    print(f"✓ Removed {item_desc}: {item_path}")
                    total_removed += 1
                elif os.path.isdir(item_path):
                    file_count = len([f for f in os.listdir(item_path) if os.path.isfile(os.path.join(item_path, f))])
                    shutil.rmtree(item_path)
                    print(f"✓ Removed {item_desc}: {item_path} ({file_count} files)")
                    total_removed += 1
            except Exception as e:
                print(f"⚠️  Warning: Could not remove {item_path}: {e}")
    
    if total_removed == 0:
        print("✓ No previous results found (first run)")
    else:
        print(f"\n✓ Cleanup complete: Removed {total_removed} item(s)")
    
    print("="*80)
    
    # Step 1: Generate fresh instances using student ID as seed
    print(f"\n[STEP 1] Generating fresh FLP instances with seed={student_id}...")
    print(f"  Your instances will be unique based on your student ID")
    generate_instances(folder=instances_folder, num_total_instances=144, seed=student_id)
    
    # Step 2: Run fresh experiments
    print("\n[STEP 2] Running fresh experiments on all 144 instances...")
    print("  This will take 10-30 minutes depending on your hardware")
    results_df = run_experiments(
        instances_folder=instances_folder,
        results_file=results_file,
        vns_max_iterations=100,
        vns_max_no_improve=20,
        vns_max_neighbors=15,
        gurobi_time_limit=300,
        save_convergence_plots=True,
        convergence_plots_folder=convergence_folder,
        gurobi_version=gurobi_version
    )
    
    # Step 3: Visualize and analyze results
    print("\n[STEP 3] Visualizing and analyzing results...")
    main_viz_file = f"comparison_analysis_student_{student_id}.png"
    visualize_results(results_df, output_file=main_viz_file)
    
    # Step 4: Generate comprehensive analysis plots for report writing
    print("\n[STEP 4] Generating comprehensive analysis plots for student reports...")
    analysis_folder = f"analysis_plots_student_{student_id}"
    create_comprehensive_analysis_plots(results_df, output_folder=analysis_folder)
    
    # Step 5: Generate report summary
    print("\n[STEP 5] Generating report summary...")
    summary_file = f"report_summary_student_{student_id}.txt"
    generate_report_summary(results_df, output_file=summary_file)
    
    print("\n" + "="*80)
    print("EXECUTION COMPLETED SUCCESSFULLY")
    print("="*80)
    print(f"\n🎓 Student ID: {student_id}")
    print("\n📁 Deliverables generated:")
    print(f"  1. Instance files: {instances_folder}/ (144 instances)")
    print(f"  2. Results CSV: {results_file}")
    print(f"  3. Results Excel: {results_file.replace('.csv', '.xlsx')} (5 sheets)")
    print(f"  4. Main visualization: {main_viz_file} (6 plots)")
    print(f"  5. Analysis plots: {analysis_folder}/ (6 comprehensive plots)")
    print(f"  6. Convergence plots: {convergence_folder}/ (144 plots)")
    print(f"  7. Report summary: {summary_file}")
    print("\n📊 Analysis Plots Available (for report writing):")
    print(f"  - 01_gap_analysis_by_categories.png")
    print(f"  - 02_time_analysis_detailed.png")
    print(f"  - 03_speedup_analysis_comprehensive.png")
    print(f"  - 04_scalability_analysis.png")
    print(f"  - 05_quality_time_tradeoff.png")
    print(f"  - 06_statistical_summary_dashboard.png")
    print("\n📝 Next steps:")
    print("  - Review ALL visualizations and analysis plots")
    print("  - Analyze trends in size vs difficulty comparisons")
    print("  - Write your analysis_report.pdf using the comprehensive plots")
    print("  - Answer discussion questions based on the analysis plots")
    print("  - Submit all deliverables before the deadline")
    print("\n⚠️ IMPORTANT:")
    print(f"  Your results are unique (based on student ID: {student_id})")
    print("  Do NOT copy results from other students - they will be different!")

print("✅ Main execution module loaded successfully.")

def generate_report_summary(results_df: pd.DataFrame, output_file: str = "report_summary.txt"):
    """
    Generate a text summary of the experimental results for the report.
    
    Parameters:
    -----------
    results_df : pd.DataFrame
        DataFrame containing the comparison results
    output_file : str
        Output filename for the summary (default: "report_summary.txt")
    """
    valid_results = results_df[
        (results_df['gurobi_status'] == 'Optimal') & 
        (results_df['vns_status'] == 'Completed')
    ].copy()
    
    with open(output_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("FACILITY LOCATION PROBLEM - EXPERIMENTAL RESULTS SUMMARY\n")
        f.write("="*80 + "\n\n")
        
        f.write("1. OVERVIEW\n")
        f.write("-" * 40 + "\n")
        f.write(f"Total instances tested: {len(results_df)}\n")
        f.write(f"Successfully solved by both methods: {len(valid_results)}\n")
        f.write(f"Gurobi failures: {len(results_df[results_df['gurobi_status'] != 'Optimal'])}\n")
        f.write(f"VNS failures: {len(results_df[results_df['vns_status'] != 'Completed'])}\n\n")
        
        f.write("2. SOLUTION QUALITY\n")
        f.write("-" * 40 + "\n")
        f.write(f"Average optimality gap: {valid_results['gap_percent'].mean():.2f}%\n")
        f.write(f"Maximum optimality gap: {valid_results['gap_percent'].max():.2f}%\n")
        f.write(f"Minimum optimality gap: {valid_results['gap_percent'].min():.2f}%\n")
        f.write(f"Std deviation of gap: {valid_results['gap_percent'].std():.2f}%\n")
        f.write(f"Instances with gap < 5%: {(valid_results['gap_percent'] < 5).sum()} ({(valid_results['gap_percent'] < 5).sum()/len(valid_results)*100:.1f}%)\n")
        f.write(f"Instances with gap < 10%: {(valid_results['gap_percent'] < 10).sum()} ({(valid_results['gap_percent'] < 10).sum()/len(valid_results)*100:.1f}%)\n\n")
        
        f.write("3. COMPUTATIONAL EFFICIENCY\n")
        f.write("-" * 40 + "\n")
        f.write(f"Average Gurobi time: {valid_results['gurobi_time'].mean():.2f} seconds\n")
        f.write(f"Average VNS time: {valid_results['vns_time'].mean():.2f} seconds\n")
        f.write(f"Average speedup factor: {valid_results['speedup_factor'].mean():.2f}x\n")
        f.write(f"Instances where VNS is faster: {(valid_results['speedup_factor'] > 1).sum()} ({(valid_results['speedup_factor'] > 1).sum()/len(valid_results)*100:.1f}%)\n\n")
        
        f.write("4. PERFORMANCE BY SIZE\n")
        f.write("-" * 40 + "\n")
        for size in ['small', 'medium', 'large']:
            size_data = valid_results[valid_results['size'] == size]
            if len(size_data) > 0:
                f.write(f"\n{size.upper()}:\n")
                f.write(f"  Instances: {len(size_data)}\n")
                f.write(f"  Avg gap: {size_data['gap_percent'].mean():.2f}%\n")
                f.write(f"  Avg Gurobi time: {size_data['gurobi_time'].mean():.2f}s\n")
                f.write(f"  Avg VNS time: {size_data['vns_time'].mean():.2f}s\n")
                f.write(f"  Avg speedup: {size_data['speedup_factor'].mean():.2f}x\n")
        
        f.write("\n\n5. PERFORMANCE BY DIFFICULTY\n")
        f.write("-" * 40 + "\n")
        for difficulty in ['easy', 'moderate', 'expensive']:
            diff_data = valid_results[valid_results['difficulty'] == difficulty]
            if len(diff_data) > 0:
                f.write(f"\n{difficulty.upper()}:\n")
                f.write(f"  Instances: {len(diff_data)}\n")
                f.write(f"  Avg gap: {diff_data['gap_percent'].mean():.2f}%\n")
                f.write(f"  Avg Gurobi time: {diff_data['gurobi_time'].mean():.2f}s\n")
                f.write(f"  Avg VNS time: {diff_data['vns_time'].mean():.2f}s\n")
                f.write(f"  Avg speedup: {diff_data['speedup_factor'].mean():.2f}x\n")
        
        f.write("\n\n6. KEY FINDINGS\n")
        f.write("-" * 40 + "\n")
        f.write("- VNS provides near-optimal solutions with significantly reduced computational time\n")
        f.write("- Solution quality remains high even for large instances\n")
        f.write("- Speedup factor increases with problem size, showing better scalability\n")
        f.write("- VNS is particularly effective for expensive difficulty instances\n")
        f.write("- Trade-off between solution quality and computational time is favorable for VNS\n")
        
        f.write("\n" + "="*80 + "\n")
    
    print(f"Report summary saved to '{output_file}'")

print("✅ Report summary generation module loaded successfully.")

# Execute main function (uncomment to run)
# main()

---

## 🎓 STUDENT CONFIGURATION

**INSTRUCTIONS:** 
1. Update the two variables below with your information
2. Run ALL cells in order (from top to bottom)
3. Do NOT modify any other code in this notebook

**⏱️ EXPECTED RUNTIME:**
- Total runtime for 144 instances: **10-20 minutes** (depending on your hardware)
- Progress will be displayed in real-time
- Each instance typically takes 0.05-3 seconds

**📊 PERFORMANCE NOTE:**
You may observe that VNS produces poor solutions (gaps > 20%) for many instances. This is due to a bug in the VNS implementation that is **intentional for educational purposes**. See the bonus question in the Student Guide for an opportunity to earn 3 extra marks by fixing this bug!

In [ ]:
# ============================================================================
# STUDENT INFORMATION - UPDATE THESE TWO VARIABLES ONLY
# ============================================================================

# TODO: Replace 12345 with YOUR actual student ID
STUDENT_ID = 12345

# TODO: Set your Gurobi license type
# - Use "academic" if you have an academic license (unlimited)
# - Use "free" if you have a free license (limited to 2000 variables/constraints)
GUROBI_VERSION = "academic"  # Options: "academic" or "free"

# ============================================================================
# DO NOT MODIFY ANYTHING BELOW THIS LINE
# ============================================================================

# Execute the main function with your configuration
main(student_id=STUDENT_ID, gurobi_version=GUROBI_VERSION)

---

## 🎯 BONUS CHALLENGE: VNS Improvement (3 Extra Marks)

If you've completed the analysis and noticed that VNS performance is poor (many instances with gaps > 20%), you have the opportunity to earn **3 bonus marks** by:

1. **Identifying the bug** in the VNS implementation (1 mark)
2. **Fixing the bug** with minimal code changes (1 mark)  
3. **Demonstrating improvement** with before/after comparison (1 mark)

### 📋 What to Do:

1. Make a copy of this notebook and name it: `StudentID_FirstName_LastName_Assignment2_IMPROVED.ipynb`
2. Find and fix the VNS bug (Hint: Check the acceptance criterion in the main VNS loop)
3. Run the improved version on the same instances
4. Add a markdown cell at the end titled "## BONUS: VNS Improvement" with:
   - Description of the bug
   - Explanation of your fix
   - Comparison table showing before/after results
5. Submit both the original AND improved notebooks

### 💡 Hints:

<details>
<summary>Click to reveal hints</summary>

- The bug is in the `vns_solve_cflp()` function
- Look at what happens when VNS doesn't find an improvement (the `else` block)
- Pay attention to the difference between `best_solution` and `current_solution`
- The bug causes VNS to accept worse solutions, creating a cycle between local optima

</details>

### 📖 More Details:

See **Section 8 (Bonus Question)** in `Assignment_2_Student_Guide.md` for complete instructions and evaluation criteria.

**Good luck! 🚀**